# Unit 4. Regression Diagnostics

精算統計模型 (SOA Exam SRM/PA). The text of [unit04.pdf](https://github.com/chang-ye-tu/asm/blob/master/note/unit04.pdf) with every R chunk as a code cell.

<a href="https://colab.research.google.com/github/chang-ye-tu/asm/blob/master/colab/unit04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The notebook runs on the R runtime (on Colab: Runtime › Change runtime type › R). Run the setup cell first: it installs the packages and downloads the data this unit uses.

In [ ]:
# Setup: run this cell first. It installs the packages this unit uses and downloads
# its data files into data/, the paths the code below reads.
local({
  if (is.null(getOption("repos")) || identical(unname(getOption("repos")["CRAN"]), "@CRAN@"))
    options(repos = c(CRAN = "https://cloud.r-project.org"))
  os <- tryCatch(readLines("/etc/os-release"), error = function(e) character())
  code <- sub("^VERSION_CODENAME=", "", grep("^VERSION_CODENAME=", os, value = TRUE))
  if (length(code) == 1 && nzchar(code))  # Linux, e.g. Colab: prebuilt binary packages
    options(repos = c(CRAN = sprintf("https://packagemanager.posit.co/cran/__linux__/%s/latest", code)),
            HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(), paste(getRversion(),
              R.version$platform, R.version$arch, R.version$os)))
})
pkgs <- c("car", "ISLR2", "lmtest", "sandwich")
new <- setdiff(pkgs, rownames(installed.packages()))
if (length(new)) install.packages(new)
data_files <- c("data/frees/csv/Liquidity.csv", "data/frees/csv/OutlierExample.csv", "data/frees/csv/TermLife.csv", "data/frees/csv/UNLifeExpectancy.csv")
for (f in data_files) if (!file.exists(f)) {
  dir.create(dirname(f), recursive = TRUE, showWarnings = FALSE)
  url <- if (startsWith(f, "data/islr/"))
    paste0("https://www.statlearning.com/s/", basename(f)) else
    paste0("https://raw.githubusercontent.com/chang-ye-tu/asm/master/note/", f)
  download.file(url, f, quiet = TRUE, mode = "wb")
}
options(width = 62, digits = 4, scipen = 4, repr.plot.width = 6.4, repr.plot.height = 3.0)

In [ ]:
library(ISLR2)

## 1 The Iterative Approach

> **Reading.** FREES §5.1, §5.3 (opening), §5.10.1 · ISLR §3.3.3 (opening). SRM learning outcomes 2(a), 2(d).

**Remark** (The classical assumptions; Unit 2). A1–A5 are Unit 2's classical assumptions on $\mathbf{Y}=\mathbf{X}\boldsymbol{\beta}+\boldsymbol{\varepsilon}$ with $\mathbf{X}$ fixed: linearity (A1), zero-mean errors (A2), $\operatorname{var}(\boldsymbol{\varepsilon})=\sigma^2\mathbf{I}_n$ with $\sigma^2>0$ (A3), $\operatorname{rank}(\mathbf{X})=k+1$, with $n>k+1$ for inference (A4), and normal errors (A5).

**Definition 4.1** (Iterative model specification; FREES §5.1). A model is specified by iterating three stages: **formulation**, from graphs of the data and prior knowledge of the relationships; **fitting**, under the assumptions of the formulated model; and **diagnostic checking**, or data and model criticism, which compares the fitted model with the data and returns the analyst to formulation when they disagree. Inferences are drawn from the model only after the data and the model agree.

**Proposition 4.2** (What the residuals carry; FREES §5.3, §5.10.1). Let $\mathbf{X}$ satisfy A4, and let $\mathbf{Y}$ have mean $\boldsymbol{\mu}$ and covariance matrix $\boldsymbol{\Sigma}$, neither $\boldsymbol{\mu}\in\mathcal{C}(\mathbf{X})$ nor $\boldsymbol{\Sigma}=\sigma^2\mathbf{I}_n$ being assumed. Then

$$
\operatorname{\mathsf{E}}[\mathbf e]=\mathbf{M}\boldsymbol{\mu},\qquad \operatorname{var}(\mathbf e)=\mathbf{M}\boldsymbol{\Sigma}\mathbf{M} ,
$$

and $\operatorname{\mathsf{E}}[\mathbf e]=\mathbf{0}$ iff $\boldsymbol{\mu}\in\mathcal{C}(\mathbf{X})$. Under A1–A3, $\operatorname{var}(\mathbf e)=\sigma^2\mathbf{M}$: $\operatorname{\mathsf{E}}[e_i]=0$ and $\operatorname{var}(e_i)=\sigma^2(1-h_{ii})$.

**Proof.** $\mathbf e=\mathbf{M}\mathbf{Y}$ with $\mathbf{M}$ constant (Unit 2's definition of the residual maker), so $\operatorname{\mathsf{E}}[\mathbf e]=\mathbf{M}\boldsymbol{\mu}$, and by the prerequisites' proposition **variance under a linear map**, $\operatorname{var}(\mathbf e)=\mathbf{M}\boldsymbol{\Sigma}\mathbf{M}'=\mathbf{M}\boldsymbol{\Sigma}\mathbf{M}$, $\mathbf{M}$ being symmetric. If $\boldsymbol{\mu}=\mathbf{X}\mathbf{c}$ then $\mathbf{M}\boldsymbol{\mu}=\mathbf{M}\mathbf{X}\mathbf{c}=\mathbf{0}$ by Unit 2's theorem **properties of $\mathbf{H}$ and $\mathbf{M}$**; conversely $\mathbf{M}\boldsymbol{\mu}=\mathbf{0}$ says $\boldsymbol{\mu}=\mathbf{H}\boldsymbol{\mu}$, a vector of $\mathcal{C}(\mathbf{X})$. Under A1, $\boldsymbol{\mu}=\mathbf{X}\boldsymbol{\beta}\in\mathcal{C}(\mathbf{X})$; under A3, $\boldsymbol{\Sigma}=\sigma^2\mathbf{I}_n$ and $\mathbf{M}\boldsymbol{\Sigma}\mathbf{M}=\sigma^2\mathbf{M}^2=\sigma^2\mathbf{M}$, whose $(i,i)$ entry is $\sigma^2(1-h_{ii})$. ∎

**Remark** (FREES §5.3.1). A pattern in the residuals estimates $\mathbf{M}\boldsymbol{\mu}$, the part of the mean that the explanatory variables do not reach. FREES lists four patterns: unusual residuals, residuals related to other explanatory variables, heteroscedastic residuals, and residuals related to time.

**Remark** (Potential problems; ISLR §3.3.3). ISLR lists six problems of a fitted linear regression: non-linearity of the response–predictor relationship, a failure of A1; correlated errors and non-constant error variance, failures of A3; outliers and high leverage points, single observations unusual in $y$ or in $\mathbf{x}$; and collinearity, a near-failure of A4.

## 2 Residual Analysis

> **Reading.** FREES §5.3.1–5.3.3, §5.10.1–5.10.2 · ISLR §3.3.3 (non-linearity, correlation of error terms, outliers). SRM learning outcome 2(d).

### 2.1 Three scalings of a residual

**Proposition 4.3** (Deleting an observation is adding its indicator; FREES §5.3.2, §5.10.2). Let $\mathbf u_i$ be the $i$-th coordinate vector of $\mathbb{R}^n$, let $\mathbf{X}_{(i)}$ and $\mathbf{Y}_{(i)}$ be $\mathbf{X}$ and $\mathbf{Y}$ without row $i$, and suppose A4 and $h_{ii}<1$.

(i) $\mathbf{X}_{(i)}$ and $[\mathbf{X}\ \mathbf u_i]$ have full column rank, so the regression without observation $i$ has a unique coefficient vector $\hat{\boldsymbol{\beta}}_{(i)}$, with residual sum of squares $\mathrm{SSE}_{(i)}$, and the regression of $\mathbf{Y}$ on $[\mathbf{X}\ \mathbf u_i]$ has a unique coefficient vector $(\tilde{\boldsymbol{\beta}}',\hat\gamma)'$.

(ii) $\tilde{\boldsymbol{\beta}}=\hat{\boldsymbol{\beta}}_{(i)}$ and $\hat\gamma=y_i-\mathbf{x}_i'\hat{\boldsymbol{\beta}}_{(i)}$, the **deleted residual** $e_{(i)}$; the residual sum of squares of the regression on $[\mathbf{X}\ \mathbf u_i]$ is $\mathrm{SSE}_{(i)}$.

(iii) $e_{(i)}=e_i/(1-h_{ii})$ and $\hat{\boldsymbol{\beta}}_{(i)}=\hat{\boldsymbol{\beta}}-(\mathbf{X}'\mathbf{X})^{-1}\mathbf{x}_i\,e_i/(1-h_{ii})$.

(iv) $\mathrm{SSE}_{(i)}=\mathrm{SSE}-e_i^2/(1-h_{ii})$.

**Proof.** **Step 1: rank.** If $\mathbf u_i=\mathbf{X}\mathbf{c}$ then $\mathbf{H}\mathbf u_i=\mathbf{H}\mathbf{X}\mathbf{c}=\mathbf{X}\mathbf{c}=\mathbf u_i$ and $h_{ii}=\mathbf u_i'\mathbf{H}\mathbf u_i=\mathbf u_i'\mathbf u_i=1$; so $h_{ii}<1$ gives $\mathbf u_i\notin\mathcal{C}(\mathbf{X})$. If $\mathbf{X}\mathbf{v}+\gamma\mathbf u_i=\mathbf{0}$, then $\gamma\neq0$ would put $\mathbf u_i=-\mathbf{X}\mathbf{v}/\gamma$ in $\mathcal{C}(\mathbf{X})$, so $\gamma=0$, $\mathbf{X}\mathbf{v}=\mathbf{0}$ and $\mathbf{v}=\mathbf{0}$ by A4. If $\mathbf{X}_{(i)}\mathbf{v}=\mathbf{0}$, then $\mathbf{X}\mathbf{v}=(\mathbf{x}_i'\mathbf{v})\mathbf u_i$ lies in $\mathcal{C}(\mathbf{X})$, which forces $\mathbf{x}_i'\mathbf{v}=0$, so $\mathbf{X}\mathbf{v}=\mathbf{0}$ and $\mathbf{v}=\mathbf{0}$. Unit 2's theorem **the least squares estimator** gives the unique coefficient vectors.

**Step 2: the two regressions coincide.** For every $\boldsymbol{\beta}$ and $\gamma$,

$$
\|\mathbf{Y}-\mathbf{X}\boldsymbol{\beta}-\gamma\mathbf u_i\|^2=\sum_{j\neq i}(y_j-\mathbf{x}_j'\boldsymbol{\beta})^2+(y_i-\mathbf{x}_i'\boldsymbol{\beta}-\gamma)^2
\geqslant\|\mathbf{Y}_{(i)}-\mathbf{X}_{(i)}\boldsymbol{\beta}\|^2\geqslant\mathrm{SSE}_{(i)} ,
$$

with equality throughout at $\boldsymbol{\beta}=\hat{\boldsymbol{\beta}}_{(i)}$, $\gamma=y_i-\mathbf{x}_i'\hat{\boldsymbol{\beta}}_{(i)}$. This pair therefore minimises the left side, and by uniqueness it is $(\tilde{\boldsymbol{\beta}},\hat\gamma)$, with minimum $\mathrm{SSE}_{(i)}$: this is (ii).

**Step 3: the formulas.** Since $\mathbf{X}'\mathbf u_i=\mathbf{x}_i$, $\mathbf u_i'\mathbf u_i=1$ and $\mathbf u_i'\mathbf{Y}=y_i$, the normal equations of the regression on $[\mathbf{X}\ \mathbf u_i]$ are

$$
\mathbf{X}'\mathbf{X}\tilde{\boldsymbol{\beta}}+\mathbf{x}_i\hat\gamma=\mathbf{X}'\mathbf{Y} ,\qquad \mathbf{x}_i'\tilde{\boldsymbol{\beta}}+\hat\gamma=y_i .
$$

The first gives $\tilde{\boldsymbol{\beta}}=\hat{\boldsymbol{\beta}}-(\mathbf{X}'\mathbf{X})^{-1}\mathbf{x}_i\hat\gamma$; substituting in the second, $\hat y_i-h_{ii}\hat\gamma+\hat\gamma=y_i$, so $\hat\gamma(1-h_{ii})=e_i$. With Step 2 this is (iii). Unit 2's lemma **adding one explanatory variable**, with $\mathbf{z}=\mathbf u_i\notin\mathcal{C}(\mathbf{X})$, $\mathbf{z}'\mathbf{M}\mathbf{Y}=e_i$ and $\mathbf{z}'\mathbf{M}\mathbf{z}=1-h_{ii}$ (the $(i,i)$ entry of $\mathbf{M}$), gives $\mathrm{SSE}-\mathrm{SSE}_{(i)}=e_i^2/(1-h_{ii})$, which is (iv). ∎

**Remark** (FREES §5.10.2, ISLR §5.1.2). Part (iii) at observation $i$ is Unit 1's LOOCV shortcut, and part (iv) is FREES's formula (5.21); FREES derives both from the prerequisites' Sherman–Morrison lemma applied to $\mathbf{X}_{(i)}'\mathbf{X}_{(i)}=\mathbf{X}'\mathbf{X}-\mathbf{x}_i\mathbf{x}_i'$.

**Definition 4.4** (Standardised and studentised residuals; FREES §5.3.1, ISLR §3.3.3). Let $h_{ii}<1$, $n>k+2$, and let $s_{(i)}^2=\mathrm{SSE}_{(i)}/(n-k-2)$ be the residual variance of the regression without observation $i$ (Proposition 4.3). Three scalings of the residual $e_i$ are

$$
\text{(a) } \frac{e_i}{s},\qquad
\text{(b) } r_i=\frac{e_i}{s\sqrt{1-h_{ii}}},\qquad
\text{(c) } t_i=\frac{e_i}{s_{(i)}\sqrt{1-h_{ii}}} ,
$$

defined when $s>0$ and $s_{(i)}>0$. $r_i$ is the **standardised residual** of Unit 2, the residual over the estimate of its standard deviation $\sigma\sqrt{1-h_{ii}}$ (Proposition 4.2); $t_i$, the **studentised residual**, estimates $\sigma$ without observation $i$. R returns $r_i$ as `rstandard` and $t_i$ as `rstudent`.

**Remark** (FREES §5.3.1). FREES motivates (a) by the spread of the residuals: with an intercept they average $0$ (Unit 2's residual identities), so their sample variance is $\mathrm{SSE}/(n-1)=s^2(n-k-1)/(n-1)$, close to $s^2$ when $k$ is small against $n$.

**Theorem 4.5** (The studentised residual; FREES §5.3.1, §5.10.2). Let $h_{ii}<1$, $n>k+2$ and $s>0$.

(i) $\mathrm{SSE}_{(i)}=s^2(n-k-1-r_i^2)$, so $s_{(i)}^2=s^2(n-k-1-r_i^2)/(n-k-2)$ and $r_i^2\leqslant n-k-1$, with equality iff $\mathrm{SSE}_{(i)}=0$.

(ii) If $s_{(i)}>0$,

$$
t_i=r_i\sqrt{\frac{n-k-2}{n-k-1-r_i^2}} , \tag{1}
$$

a strictly increasing function of $r_i$, and $|t_i|>|r_i|$ iff $|r_i|>1$.

(iii) $t_i$ is the $t$ ratio of $\hat\gamma$ in the regression of $\mathbf{Y}$ on $[\mathbf{X}\ \mathbf u_i]$; under A1–A5, $t_i\sim t_{n-k-2}$.

(iv) If $s_{(i)}>0$ and the other responses are held fixed, then $|t_i|\to\infty$ and $r_i^2\to n-k-1$ as $|y_i|\to\infty$.

**Proof.** **(i)** By the definition of $r_i$, $e_i^2/(1-h_{ii})=r_i^2s^2$, and $\mathrm{SSE}=(n-k-1)s^2$, so Proposition 4.3(iv) reads $\mathrm{SSE}_{(i)}=s^2(n-k-1-r_i^2)$. Dividing by $n-k-2$ gives $s_{(i)}^2$, and $\mathrm{SSE}_{(i)}\geqslant0$ gives the bound, with equality iff $\mathrm{SSE}_{(i)}=0$.

**(ii)** $t_i=r_i\,s/s_{(i)}$, and by (i) $s/s_{(i)}=\sqrt{(n-k-2)/(n-k-1-r_i^2)}$. Write $m=n-k-1$ and $f(r)=r\sqrt{(m-1)/(m-r^2)}$ on $(-\sqrt m,\sqrt m)$; then

$$
f'(r)=\sqrt{m-1}\,\Bigl\{(m-r^2)^{-1/2}+r^2(m-r^2)^{-3/2}\Bigr\}
=\frac{m\sqrt{m-1}}{(m-r^2)^{3/2}}>0 .
$$

For $r_i\neq0$, $|t_i|/|r_i|=\sqrt{(m-1)/(m-r_i^2)}$ exceeds $1$ iff $m-1>m-r_i^2$, that is iff $r_i^2>1$.

**(iii)** By Proposition 4.3, the regression on $[\mathbf{X}\ \mathbf u_i]$ has $\hat\gamma=e_i/(1-h_{ii})$ and residual sum of squares $\mathrm{SSE}_{(i)}$ on $n-(k+2)$ degrees of freedom, so its residual variance is $s_{(i)}^2$, and Unit 2's lemma **adding one explanatory variable** gives $\mathrm{SE}(\hat\gamma)=s_{(i)}/\sqrt{\mathbf u_i'\mathbf{M}\mathbf u_i}=s_{(i)}/\sqrt{1-h_{ii}}$. Hence $\hat\gamma/\mathrm{SE}(\hat\gamma)=e_i/(s_{(i)}\sqrt{1-h_{ii}})=t_i$. Under A1–A5, $\mathbf{Y}=\mathbf{X}\boldsymbol{\beta}+0\cdot\mathbf u_i+\boldsymbol{\varepsilon}$ satisfies A1–A5 for the design $[\mathbf{X}\ \mathbf u_i]$, which has full column rank $k+2<n$ by Proposition 4.3(i). Unit 2's theorem **the $t$ statistic** with $\mathbf{c}$ the last coordinate vector and true value $\gamma=0$ gives $t_i\sim t_{n-k-2}$.

**(iv)** $\mathbf e=\mathbf{M}\mathbf{Y}$, so $e_i=(1-h_{ii})y_i-\sum_{j\neq i}h_{ij}y_j$ is affine in $y_i$ with slope $1-h_{ii}>0$, and $|e_i|\to\infty$. $\mathrm{SSE}_{(i)}$ and $s_{(i)}$ do not involve $y_i$, so $|t_i|\to\infty$. By Proposition 4.3(iv), $\mathrm{SSE}=\mathrm{SSE}_{(i)}+E_i$ with $E_i=e_i^2/(1-h_{ii})$, and $r_i^2=E_i/s^2=(n-k-1)E_i/(\mathrm{SSE}_{(i)}+E_i)\to n-k-1$ as $E_i\to\infty$. ∎

**Remark** (FREES §5.3.1, ISLR §3.3.3). Part (iv) is FREES's argument for $t_i$: an unusually large response inflates $s$ together with $e_i$, so $|r_i|$ can never pass $\sqrt{n-k-1}$, while $s_{(i)}$ is untouched and $|t_i|$ grows without bound. FREES gives $t_i$ the $t$ law on $n-(k+1)$ degrees of freedom; part (iii) shows the count is $n-(k+1)-1$, since $s_{(i)}$ comes from $n-1$ observations and $k+1$ coefficients. ISLR defines the studentised residual as the residual over its estimated standard error, which describes $r_i$ and $t_i$ alike; its lab computes $t_i$.

**Corollary 4.6** (The largest studentised residual (beyond the syllabus)). Suppose A1–A5, $n>k+2$ and $h_{ii}<1$ for every $i$, and let $c=t_{n-k-2,\,1-\alpha/(2n)}$. Then $\operatorname{\mathsf{P}}\{\max_i|t_i|>c\}\leqslant\alpha$.

**Proof.** $\{\max_i|t_i|>c\}=\bigcup_{i=1}^n\{|t_i|>c\}$. By subadditivity of probability and Theorem 4.5(iii), with the $t$ law symmetric, $\operatorname{\mathsf{P}}\{\max_i|t_i|>c\}\leqslant\sum_{i=1}^n\operatorname{\mathsf{P}}\{|t_i|>c\}=n\cdot2\cdot\alpha/(2n)=\alpha$. No independence among the $t_i$ is used. ∎

**Example 4.1** (Outliers and high leverage points, continued; FREES §5.3.1). The $19$ base points of Unit 2's example with one of the unusual points A, B, C added, so $n=20$ and $k=1$:

In [ ]:
ol <- read.csv("data/frees/csv/OutlierExample.csv")
abc <- sapply(1:3, function(code) {
  d <- subset(ol, CODES %in% c(0, code)); m <- lm(Y ~ X, data = d); i <- nrow(d)
  r <- rstandard(m)[[i]]
  c(r = r, t = rstudent(m)[[i]], t_from_r = r * sqrt((20 - 1 - 2) / (20 - 1 - 1 - r^2)),
    s = summary(m)$sigma, s_without = summary(lm(Y ~ X, data = d[-i, ]))$sigma)
})
colnames(abc) <- c("A", "B", "C"); round(abc, 4)
c(ceiling_r = sqrt(20 - 1 - 1), bonferroni_cut = qt(1 - 0.05 / (2 * 20), 20 - 1 - 2))

A and C have $r=4.00$ and $-4.01$, close to the ceiling $\sqrt{n-k-1}=4.24$ of Theorem 4.5(i). Without them the fit to the base points has $s_{(i)}=0.288$ against $s=0.846$ and $0.865$ with them, so their studentised residuals are $11.74$ and $-12.04$, far beyond the cutoff $3.54$ of Corollary 4.6 at $\alpha=0.05$. B has $|r|<1$, and its $t$ is smaller still, as Theorem 4.5(ii) requires.

### 2.2 Outliers

**Proposition 4.7** (How often a correct model is flagged; FREES §5.3.2). Under A1–A5 with $n>k+5$ and $h_{jj}<1$ for every $j$,

$$
\operatorname{\mathsf{P}}\{|r_i|>2\}=\operatorname{\mathsf{P}}\Bigl\{|T|>2\sqrt{\tfrac{n-k-2}{n-k-5}}\Bigr\},\qquad T\sim t_{n-k-2},
$$

and the expected number of observations with $|r_i|>2$ is $n$ times this probability.

**Proof.** By Theorem 4.5(ii), $r\mapsto t$ is strictly increasing and odd, so $|r_i|>2$ iff $|t_i|>2\sqrt{(n-k-2)/(n-k-1-4)}$; Theorem 4.5(iii) gives the law of $t_i$. The count is $\sum_i\mathbb{1}\{|r_i|>2\}$, whose expectation is the sum of the $n$ equal probabilities. ∎

In [ ]:
flag <- function(n, k) 2 * pt(-2 * sqrt((n - k - 2) / (n - k - 5)), n - k - 2)
round(c(n20_k1 = flag(20, 1), n123_k2 = flag(123, 2), n400_k2 = flag(400, 2),
        normal = 2 * pnorm(-2), expected_flags_n400 = 400 * flag(400, 2)), 4)

**Remark** (Flagging outliers; FREES §5.3.2, ISLR §3.3.3). FREES flags observation $i$ as an **outlier** when $|r_i|>2$, and ISLR when its studentised residual exceeds $3$ in absolute value. By Proposition 4.7 a correct model with $n=400$ and $k=2$ has each $|r_i|>2$ with probability $0.045$, about $18$ flags expected, and a flag is a reason to investigate the observation, not evidence against the model.

**Remark** (Options for handling outliers; FREES §5.3.2). FREES's options are to keep the observation and comment on its effect; to delete it as unrepresentative and report it separately; or to add an indicator variable for it, which formalises a special cause that later observations may share. By Proposition 4.3 and Theorem 4.5(iii), the indicator reproduces the deletion exactly: the same coefficients for the other variables, the same residuals and $\mathrm{SSE}$ for the other observations, and an exact fit at the outlier, whose indicator coefficient is the deleted residual and whose $t$ ratio is $t_i$.

### 2.3 Residuals and other explanatory variables

**Proposition 4.8** (Residuals against a candidate variable; FREES §5.3.3, §5.10.1). Suppose A4, that the model has an intercept and $\mathrm{SSE}>0$, write $r(\cdot,\cdot)$ for a sample correlation, and let $\mathbf{v}\notin\mathcal{C}(\mathbf{X})$ be a candidate explanatory variable whose regression on $\mathbf{X}$ has coefficient of determination $R_v^2$.

(i) $r(\mathbf e,\mathbf{x}_j)=0$ for every non-constant explanatory variable $x_j$ in the model, and $r(\mathbf e,\hat{\mathbf{Y}})=0$ if $\hat{\mathbf{Y}}$ is not constant.

(ii) $r(\mathbf e,\mathbf{v})=r(y,v\mid x_1,\ldots,x_k)\sqrt{1-R_v^2}$, so $|r(\mathbf e,\mathbf{v})|\leqslant|r(y,v\mid x_1,\ldots,x_k)|$.

(iii) If the regression on $[\mathbf{X}\ \mathbf{v}]$ has positive residual sum of squares, the $t$ ratio $t_v$ of $v$ in it satisfies $r(y,v\mid x_1,\ldots,x_k)=t_v/\sqrt{t_v^2+n-k-2}$.

**Proof.** For a vector $\mathbf{w}$ with $\mathbf{M}_0\mathbf{w}\neq\mathbf{0}$, where $\mathbf{M}_0=\mathbf{I}-\tfrac1n\mathbf{1}\mathbf{1}'$, $r(\mathbf e,\mathbf{w})=\mathbf e'\mathbf{M}_0\mathbf{w}/(\|\mathbf{M}_0\mathbf e\|\,\|\mathbf{M}_0\mathbf{w}\|)$. By Unit 2's residual identities $\mathbf{1}'\mathbf e=0$, so $\mathbf{M}_0\mathbf e=\mathbf e$ and $\mathbf e'\mathbf{M}_0\mathbf{w}=\mathbf e'\mathbf{w}$.

**(i)** $\mathbf{X}'\mathbf e=\mathbf{0}$ and $\hat{\mathbf{Y}}'\mathbf e=0$ by the same identities.

**(ii)** In Unit 3's added variable plot of $v$ in the regression on $[\mathbf{X}\ \mathbf{v}]$, the coordinates are $\mathbf{M}\mathbf{Y}=\mathbf e$ and $\mathbf{M}\mathbf{v}$, both with mean zero since $\mathbf{1}\in\mathcal{C}(\mathbf{X})$, so $r(y,v\mid x_1,\ldots,x_k)=\mathbf e'\mathbf{M}\mathbf{v}/(\|\mathbf e\|\,\|\mathbf{M}\mathbf{v}\|)$, and $\mathbf{M}\mathbf{v}\neq\mathbf{0}$ because $\mathbf{v}\notin\mathcal{C}(\mathbf{X})$. Since $\mathbf{M}$ is symmetric and $\mathbf{M}\mathbf e=\mathbf{M}^2\mathbf{Y}=\mathbf e$, $\mathbf e'\mathbf{v}=\mathbf e'\mathbf{M}\mathbf{v}$. Hence

$$
r(\mathbf e,\mathbf{v})=\frac{\mathbf e'\mathbf{M}\mathbf{v}}{\|\mathbf e\|\,\|\mathbf{M}_0\mathbf{v}\|}
=r(y,v\mid x_1,\ldots,x_k)\,\frac{\|\mathbf{M}\mathbf{v}\|}{\|\mathbf{M}_0\mathbf{v}\|} ,
$$

where $\|\mathbf{M}\mathbf{v}\|^2$ and $\|\mathbf{M}_0\mathbf{v}\|^2>0$ are the residual and total sums of squares of the regression of $\mathbf{v}$ on $\mathbf{X}$, whose ratio is $1-R_v^2$ by Unit 2's variance decomposition.

**(iii)** Unit 3's proposition **partial correlation from the $t$ ratio**, applied to the regression on $[\mathbf{X}\ \mathbf{v}]$, which has full column rank because $\mathbf{v}\notin\mathcal{C}(\mathbf{X})$, $k+1$ explanatory variables and $n-k-2$ residual degrees of freedom. ∎

**Remark** (FREES §5.3.3). By (ii), the correlation of the residuals with a candidate understates the partial correlation that governs its $t$ ratio by the factor $\sqrt{1-R_v^2}$, severely when the candidate is nearly a combination of the variables already in the model; Unit 3's added variable plot shows the partial correlation itself. A correlation also misses curvature: residuals $\propto v^2-\tfrac23$ against $v\in\{-1,0,1\}$ lie on a parabola and have correlation $0$, which a scatter plot reveals.

**Example 4.2** (Stock market liquidity; FREES §5.3.3). Trading volume `VOLUME` (millions of shares over three months) of $123$ companies, with the number of transactions `NTRAN`, the average time between transactions `AVGT` (minutes), the opening price `PRICE`, the shares outstanding `SHARE` (millions), the market value $\texttt{VALUE}\approx\texttt{PRICE}\times\texttt{SHARE}/1000$ (billions of dollars) and the debt-to-equity ratio `DEBEQ`; FREES removed three companies with extreme volume or price before the analysis. FREES Tables 5.2 and 5.3:

In [ ]:
liq <- read.csv("data/frees/csv/Liquidity.csv"); nrow(liq)
v7 <- c("VOLUME", "AVGT", "NTRAN", "PRICE", "SHARE", "VALUE", "DEBEQ")
round(t(sapply(liq[v7], function(x)
  c(mean = mean(x), median = median(x), sd = sd(x), min = min(x), max = max(x)))), 3)
round(cor(liq[v7]), 3)
c(cor_AVGT_1overNTRAN = cor(liq$AVGT, 1 / liq$NTRAN))

In [ ]:
options(repr.plot.width = 4.2, repr.plot.height = 2.9)
pairs(liq[v7], pch = 16, cex = 0.3, col = "grey30", gap = 0.25, cex.labels = 0.8)

*Figure 4.1.* The scatterplot matrix of the seven liquidity variables (FREES Figure 5.2): the curved relation of `AVGT` to `NTRAN` and the few extreme companies are visible, which the correlation matrix alone does not show.

`AVGT` and $1/\texttt{NTRAN}$ have correlation $0.9998$: the average time between trades times the number of trades is nearly the length of the trading period. The first fit and FREES Table 5.4, the correlations of its residuals with the other variables:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
m1 <- lm(VOLUME ~ NTRAN, data = liq)
round(coef(summary(m1)), 6)
c(R2 = summary(m1)$r.squared, s = summary(m1)$sigma)
cand <- c("AVGT", "PRICE", "SHARE", "VALUE", "DEBEQ")
round(rbind(residual = drop(cor(residuals(m1), liq[cand])),
            standardised = drop(cor(rstandard(m1), liq[cand]))), 3)

FREES's Table 5.4 prints the second row, the correlations of the standardised residuals. `AVGT` has the largest correlation, $-0.159$, and enters next:

In [ ]:
m2 <- lm(VOLUME ~ NTRAN + AVGT, data = liq)
round(coef(summary(m2)), 6)
c(R2 = summary(m2)$r.squared, s = summary(m2)$sigma)
round(rbind(residual = drop(cor(residuals(m2), liq[cand[-1]])),
            standardised = drop(cor(rstandard(m2), liq[cand[-1]]))), 3)

The fitted model is $\widehat{\texttt{VOLUME}}=4.41-0.322\,\texttt{AVGT} +0.00167\,\texttt{NTRAN}$ with $t(b_{\texttt{AVGT}})=-2.39$, $R^2=84.2\%$ and $s=4.26$, against $83.4\%$ and $4.35$ for the first model; FREES's Table 5.5 is the standardised row. Proposition 4.8(ii) links the first model's residual correlation with `AVGT`'s $t$ ratio in the second, and recovers the partial correlation of every remaining candidate from the second model's residuals:

In [ ]:
t_A <- coef(summary(m2))["AVGT", "t value"]; n <- nrow(liq)
R2_A <- summary(lm(AVGT ~ NTRAN, data = liq))$r.squared
c(partial_from_t = t_A / sqrt(t_A^2 + n - 3), sqrt_1_minus_R2v = sqrt(1 - R2_A),
  product = t_A / sqrt(t_A^2 + n - 3) * sqrt(1 - R2_A),
  cor_resid_AVGT = cor(residuals(m1), liq$AVGT))
round(sapply(cand[-1], function(v) {
  R2v <- summary(lm(reformulate(c("NTRAN", "AVGT"), v), data = liq))$r.squared
  tv <- coef(summary(lm(reformulate(c("NTRAN", "AVGT", v), "VOLUME"), data = liq)))[v, 3]
  c(cor_resid = cor(residuals(m2), liq[[v]]), R2v = R2v,
    partial = cor(residuals(m2), liq[[v]]) / sqrt(1 - R2v),
    partial_from_t = tv / sqrt(tv^2 + n - 4), t = tv)
}), 4)

`SHARE` and `VALUE` have residual correlations $0.100$ and $0.074$, but $R_v^2=0.69$ and $0.64$ turn these into partial correlations $0.180$ and $0.124$, and `SHARE` would enter with $t=1.99$. The candidates are strongly skewed:

In [ ]:
skew <- function(x) mean((x - mean(x))^3) / (mean((x - mean(x))^2))^1.5
round(sapply(c(liq[c("PRICE", "SHARE", "VALUE", "DEBEQ")], list(resid = residuals(m2))), skew), 2)

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.2)
op <- par(mfrow = c(1, 4), mar = c(3.2, 2.6, 0.6, 0.4), mgp = c(1.9, 0.6, 0))
for (v in c("PRICE", "SHARE", "VALUE", "DEBEQ"))
  plot(liq[[v]], residuals(m2), xlab = v, ylab = if (v == "PRICE") "residual" else "",
       pch = 16, cex = 0.45)
par(op)

*Figure 4.2.* Residuals of `VOLUME` on `NTRAN` and `AVGT` against the four remaining candidates (FREES Figure 5.3).

The residuals have skewness $0.60$, while `SHARE`, `VALUE` and `DEBEQ` have skewness $3.18$, $6.26$ and $5.42$; their plots are clouds compressed against the left edge, and no plot shows a trend.

**Example 4.3** (Non-linearity; ISLR §3.3.3). The regression of `mpg` on `horsepower` for the $392$ cars of `Auto`, with and without the square of `horsepower` (ISLR Figure 3.9):

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.4)
nrow(Auto)
a1 <- lm(mpg ~ horsepower, data = Auto)
a2 <- lm(mpg ~ horsepower + I(horsepower^2), data = Auto)
op <- par(mfrow = c(1, 2), mar = c(3.2, 3.4, 1.4, 0.6), mgp = c(1.9, 0.6, 0))
for (m in list(a1, a2)) {
  plot(fitted(m), residuals(m), xlab = "fitted", ylab = "residual", pch = 16, cex = 0.4,
       main = if (identical(m, a1)) "linear" else "quadratic", cex.main = 0.9)
  lines(lowess(fitted(m), residuals(m)), col = "#B45309", lwd = 2); abline(h = 0, lty = 2)
}
par(op)

*Figure 4.3.* Residuals against fitted values, with a `lowess` smooth, for the linear and the quadratic fit of `mpg` on `horsepower`.

The linear fit leaves a U: positive residuals at both ends of the fitted range and negative ones in the middle, the estimate of $\mathbf{M}\boldsymbol{\mu}$ of Proposition 4.2 for a curved mean. The quadratic fit's smooth stays near $0$ except at the largest fitted values. With several explanatory variables the residuals are plotted against the fitted values, with which they have correlation $0$ (Proposition 4.8(i)), so any trend in the plot is non-linear.

### 2.4 Correlated errors

**Proposition 4.9** (Least squares under a general error covariance; ISLR §3.3.3, CLM exercise B5). Assume A1, A2 and A4, and let $\operatorname{var}(\boldsymbol{\varepsilon})=\boldsymbol{\Sigma}$ be any covariance matrix. Then $\operatorname{\mathsf{E}}[\hat{\boldsymbol{\beta}}]=\boldsymbol{\beta}$,

$$
\operatorname{var}(\hat{\boldsymbol{\beta}})=(\mathbf{X}'\mathbf{X})^{-1}\mathbf{X}'\boldsymbol{\Sigma}\mathbf{X}(\mathbf{X}'\mathbf{X})^{-1},\qquad
\operatorname{\mathsf{E}}[s^2]=\frac{\operatorname{tr}(\mathbf{M}\boldsymbol{\Sigma})}{n-k-1},
$$

so the reported $s^2(\mathbf{X}'\mathbf{X})^{-1}$ has expectation $\{\operatorname{tr}(\mathbf{M}\boldsymbol{\Sigma})/(n-k-1)\}(\mathbf{X}'\mathbf{X})^{-1}$.

**Proof.** $\hat{\boldsymbol{\beta}}-\boldsymbol{\beta}=(\mathbf{X}'\mathbf{X})^{-1}\mathbf{X}'\boldsymbol{\varepsilon}$ has mean $\mathbf{0}$ by A2, and the variance follows from the prerequisites' proposition **variance under a linear map**. By Unit 2's identity $\mathbf e=\mathbf{M}\boldsymbol{\varepsilon}$ and the symmetry and idempotency of $\mathbf{M}$, $\mathrm{SSE}=\boldsymbol{\varepsilon}'\mathbf{M}\boldsymbol{\varepsilon}$, and the prerequisites' lemma **expectation of a quadratic form** gives $\operatorname{\mathsf{E}}[\mathrm{SSE}]=\operatorname{tr}(\mathbf{M}\boldsymbol{\Sigma})$. Divide by $n-k-1$. ∎

**Corollary 4.10** (Equicorrelated errors; ISLR §3.3.3). In the model $Y_i=\beta_0+\varepsilon_i$, with $k=0$ and $\hat\beta_0=\bar Y$, let $\operatorname{var}(\varepsilon_i)=\sigma^2$ and $\operatorname{cor}(\varepsilon_i,\varepsilon_j)=\rho$ for $i\neq j$, with $-1/(n-1)<\rho<1$, so that the covariance matrix is positive definite. Then

$$
\operatorname{var}(\bar Y)=\frac{\sigma^2}{n}\{1+(n-1)\rho\},\qquad
\operatorname{\mathsf{E}}\Bigl[\frac{s^2}{n}\Bigr]=\frac{\sigma^2}{n}(1-\rho) ,
$$

so the reported variance of $\bar Y$ has expectation $(1-\rho)/\{1+(n-1)\rho\}$ times the true one, a factor below $1$ for $\rho>0$ that tends to $0$ as $n\to\infty$.

**Proof.** $\boldsymbol{\Sigma}=\sigma^2\{(1-\rho)\mathbf{I}_n+\rho\mathbf{1}\mathbf{1}'\}$ and $\mathbf{X}=\mathbf{1}$, so $\operatorname{var}(\bar Y)=n^{-2}\mathbf{1}'\boldsymbol{\Sigma}\mathbf{1}=n^{-2}\sigma^2\{(1-\rho)n+\rho n^2\}$, as in Unit 1's proposition **mean of correlated estimates**. Here $\mathbf{M}=\mathbf{M}_0=\mathbf{I}-\tfrac1n\mathbf{1}\mathbf{1}'$ with $\mathbf{M}_0\mathbf{1}=\mathbf{0}$ and $\operatorname{tr}(\mathbf{M}_0)=n-1$, so $\operatorname{tr}(\mathbf{M}_0\boldsymbol{\Sigma})=\sigma^2(1-\rho)(n-1)$ and Proposition 4.9 gives $\operatorname{\mathsf{E}}[s^2]=\sigma^2(1-\rho)$. ∎

**Proposition 4.11** (Doubled data; ISLR §3.3.3). Stack the data on themselves, $\mathbf{X}_d=\begin{pmatrix}\mathbf{X}\\\mathbf{X}\end{pmatrix}$ and $\mathbf{Y}_d=\begin{pmatrix}\mathbf{Y}\\\mathbf{Y}\end{pmatrix}$, and suppose A4. The least squares coefficients are unchanged, and every standard error computed from the $2n$ rows equals the original one times $\sqrt{(n-k-1)/(2n-k-1)}$, which tends to $1/\sqrt2$.

**Proof.** $\mathbf{X}_d'\mathbf{X}_d=2\mathbf{X}'\mathbf{X}$ and $\mathbf{X}_d'\mathbf{Y}_d=2\mathbf{X}'\mathbf{Y}$, so the normal equations are those of the original data times $2$. The residual vector is $\mathbf e$ stacked twice, so $\mathrm{SSE}_d=2\mathrm{SSE}$ and $s_d^2=2\mathrm{SSE}/(2n-k-1)$, and

$$
\mathrm{SE}_d(\hat\beta_j)=\sqrt{\frac{2\mathrm{SSE}}{2n-k-1}}\sqrt{\frac{[(\mathbf{X}'\mathbf{X})^{-1}]_{jj}}{2}}
=\sqrt{\frac{n-k-1}{2n-k-1}}\;\sqrt{\frac{\mathrm{SSE}}{n-k-1}}\sqrt{[(\mathbf{X}'\mathbf{X})^{-1}]_{jj}} .
$$

∎

**Remark** (ISLR §3.3.3). The duplicated errors are perfectly correlated in pairs, so $\boldsymbol{\Sigma}$ violates A3, and the standard errors behave as if there were $2n$ independent observations. Positive correlation between the errors of adjacent observations, common in time series and in clustered data, shows as **tracking** in a plot of the residuals against time: neighbouring residuals take similar values.

**Example 4.4** (Tracking; ISLR §3.3.3). The simple regression on $x_t=t$, $t=1,\ldots,100$, with errors of variance $1$ and $\operatorname{cor}(\varepsilon_t,\varepsilon_u)=\rho^{|t-u|}$. The exact variance of $b_1$ and the exact expectation of its reported variance come from Proposition 4.9; the coverage of the nominal $95\%$ interval for $\beta_1$ is simulated:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
set.seed(4); nT <- 100; X <- cbind(1, 1:nT); XtXi <- solve(crossprod(X))
M <- diag(nT) - X %*% XtXi %*% t(X)
track <- t(sapply(c(0, 0.5, 0.9), function(rho) {
  S <- rho^abs(outer(1:nT, 1:nT, "-")); L <- t(chol(S))
  true_var <- (XtXi %*% t(X) %*% S %*% X %*% XtXi)[2, 2]
  rep_var <- sum(diag(M %*% S)) / (nT - 2) * XtXi[2, 2]
  cover <- mean(replicate(2000, {
    y <- 1 + 0.1 * (1:nT) + drop(L %*% rnorm(nT)); f <- lm.fit(X, y)
    se <- sqrt(sum(f$residuals^2) / (nT - 2) * XtXi[2, 2])
    abs(f$coefficients[2] - 0.1) <= qt(0.975, nT - 2) * se
  }))
  c(rho = rho, true_over_reported = true_var / rep_var, coverage = cover)
}))
round(track, 3)

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.2)
set.seed(7)
op <- par(mfrow = c(1, 3), mar = c(3.2, 3.2, 1.4, 0.4), mgp = c(1.9, 0.6, 0))
for (rho in c(0, 0.5, 0.9)) {
  L <- t(chol(rho^abs(outer(1:nT, 1:nT, "-")))); y <- 1 + 0.1 * (1:nT) + drop(L %*% rnorm(nT))
  plot(1:nT, residuals(lm(y ~ I(1:nT))), type = "b", pch = 16, cex = 0.35, xlab = "t",
       ylab = if (rho == 0) "residual" else "", main = bquote(rho == .(rho)), cex.main = 0.9)
  abline(h = 0, lty = 2)
}
par(op)

*Figure 4.4.* Residuals against time for $\rho=0$, $0.5$ and $0.9$ (ISLR Figure 3.10).

At $\rho=0.5$ the true variance of $b_1$ is $3.0$ times the expectation of its reported variance and the nominal $95\%$ interval covers $\beta_1$ in $75\%$ of the samples; at $\rho=0.9$ the factor is $19.6$ and the coverage $35\%$. The residual series wander in longer runs as $\rho$ grows.

## 3 Influential Points

> **Reading.** FREES §5.4, §5.10.2 · ISLR §3.3.3 (high leverage points). SRM learning outcome 2(d).

### 3.1 Leverage

**Proposition 4.12** (Leverage as a distance; FREES §5.4.1, ISLR §3.3.3). Suppose the model has an intercept, write $\mathbf{X}=[\mathbf{1}\ \tilde{\mathbf{X}}]$ and $\mathbf{x}_0=(1,\tilde{\mathbf{x}}_0')'$ for a point with $\tilde{\mathbf{x}}_0\in\mathbb{R}^k$, let $\bar{\mathbf{x}}$ be the vector of means of the $k$ explanatory variables, $\tilde{\mathbf{X}}_c=\tilde{\mathbf{X}}-\mathbf{1}\bar{\mathbf{x}}'$, and $\mathbf{S}=\tilde{\mathbf{X}}_c'\tilde{\mathbf{X}}_c/(n-1)$ their sample covariance matrix. Under A4, $\mathbf{S}$ is positive definite,

$$
\mathbf{H}=\tfrac1n\mathbf{1}\mathbf{1}'+\tilde{\mathbf{X}}_c(\tilde{\mathbf{X}}_c'\tilde{\mathbf{X}}_c)^{-1}\tilde{\mathbf{X}}_c' , \tag{2}
$$

and

$$
\mathbf{x}_0'(\mathbf{X}'\mathbf{X})^{-1}\mathbf{x}_0=\frac1n+\frac{(\tilde{\mathbf{x}}_0-\bar{\mathbf{x}})'\mathbf{S}^{-1}(\tilde{\mathbf{x}}_0-\bar{\mathbf{x}})}{n-1} . \tag{3}
$$

In particular $h_{ii}=1/n+\Delta_i^2/(n-1)$, where $\Delta_i^2=(\tilde{\mathbf{x}}_i-\bar{\mathbf{x}})'\mathbf{S}^{-1}(\tilde{\mathbf{x}}_i-\bar{\mathbf{x}})$ is the squared **Mahalanobis distance** of observation $i$ from the centre of the design; and under A1–A4 the fitted mean $\mathbf{x}_0'\hat{\boldsymbol{\beta}}$ has variance $\sigma^2\mathbf{x}_0'(\mathbf{X}'\mathbf{X})^{-1}\mathbf{x}_0$.

**Proof.** Let $\mathbf{A}=\begin{pmatrix}1&-\bar{\mathbf{x}}'\\\mathbf{0}&\mathbf{I}_k\end{pmatrix}$, invertible, so that $\mathbf{X}\mathbf{A}=[\mathbf{1}\ \ \tilde{\mathbf{X}}-\mathbf{1}\bar{\mathbf{x}}']=[\mathbf{1}\ \tilde{\mathbf{X}}_c]$. The columns of $\tilde{\mathbf{X}}_c$ sum to zero, so $(\mathbf{X}\mathbf{A})'(\mathbf{X}\mathbf{A})=\operatorname{diag}\bigl(n,\ \tilde{\mathbf{X}}_c'\tilde{\mathbf{X}}_c\bigr)$. By A4 and the invertibility of $\mathbf{A}$, $\mathbf{X}\mathbf{A}$ has full column rank, so this matrix is invertible and so is $\tilde{\mathbf{X}}_c'\tilde{\mathbf{X}}_c=(n-1)\mathbf{S}$; being a Gram matrix it is positive semi-definite, $\mathbf{v}'\tilde{\mathbf{X}}_c'\tilde{\mathbf{X}}_c\mathbf{v}=\|\tilde{\mathbf{X}}_c\mathbf{v}\|^2$, and an invertible positive semi-definite matrix is positive definite. From $(\mathbf{X}\mathbf{A})'(\mathbf{X}\mathbf{A})=\mathbf{A}'\mathbf{X}'\mathbf{X}\mathbf{A}$,

$$
(\mathbf{X}'\mathbf{X})^{-1}=\mathbf{A}\,\operatorname{diag}\bigl(1/n,\ (\tilde{\mathbf{X}}_c'\tilde{\mathbf{X}}_c)^{-1}\bigr)\mathbf{A}' .
$$

Pre- and post-multiplying by $\mathbf{X}$ and $\mathbf{X}'$ gives (2), since $\mathbf{X}\mathbf{A}=[\mathbf{1}\ \tilde{\mathbf{X}}_c]$. For (3), $\mathbf{A}'\mathbf{x}_0=(1,(\tilde{\mathbf{x}}_0-\bar{\mathbf{x}})')'$, so $\mathbf{x}_0'(\mathbf{X}'\mathbf{X})^{-1}\mathbf{x}_0=1/n+(\tilde{\mathbf{x}}_0-\bar{\mathbf{x}})'\{(n-1)\mathbf{S}\}^{-1}(\tilde{\mathbf{x}}_0-\bar{\mathbf{x}})$. Taking $\mathbf{x}_0=\mathbf{x}_i$ gives $h_{ii}$. Finally $\operatorname{var}(\mathbf{x}_0'\hat{\boldsymbol{\beta}})=\mathbf{x}_0'\operatorname{var}(\hat{\boldsymbol{\beta}})\mathbf{x}_0=\sigma^2\mathbf{x}_0'(\mathbf{X}'\mathbf{X})^{-1}\mathbf{x}_0$ by Unit 2's theorem **variance–covariance matrix**. ∎

**Remark** (FREES §5.4.1, ISLR §3.3.3). For $k=1$, (3) is Unit 2's formula $h_{ii}=1/n+(x_i-\bar x)^2/\sum_j(x_j-\bar x)^2$. The distance $\Delta_i$ weighs the explanatory variables jointly, so a point can be ordinary in each variable and unusual in the design: for two variables with unit variances and correlation $r$, the point one standard deviation above the mean in the first and one below in the second has $\Delta^2=(1,-1)\mathbf{S}^{-1}(1,-1)'=2(1+r)/(1-r^2)=2/(1-r)$, which is $20$ at $r=0.9$ (FREES Figure 5.4, ISLR Figure 3.13).

**Remark** (High leverage points; FREES §5.4.1, ISLR §3.3.3). With an intercept, Unit 2's theorem **properties of leverage** gives $1/n\leqslant h_{ii}\leqslant1$ and the mean leverage $(k+1)/n$. FREES declares observation $i$ a **high leverage point** when $h_{ii}>3(k+1)/n$, three times the mean; ISLR when $h_{ii}$ greatly exceeds $(k+1)/n$. Leverage depends on $\mathbf{X}$ alone, so replacing the response by a transformation of it leaves every leverage unchanged.

**Remark** (Options for handling high leverage points; FREES §5.4.1). FREES's options: keep the observation and comment on its effect; delete it as unrepresentative, or present the analysis with and without it; replace the explanatory variable by another carrying similar information; or transform the variable, for example replacing `DEBEQ` by the indicator of $\texttt{DEBEQ}>5$, which bounds the distance of every company in that coordinate. Robust estimation, which down-weights unusual observations, is FREES's intermediate course.

### 3.2 Cook's distance

**Definition 4.13** (Cook's distance; FREES §5.4.2). For $h_{ii}<1$ and $s>0$, with $\hat{\mathbf{Y}}_{(i)}=\mathbf{X}\hat{\boldsymbol{\beta}}_{(i)}$ the fitted values of all $n$ observations from the regression without observation $i$,

$$
D_i=\frac{\|\hat{\mathbf{Y}}-\hat{\mathbf{Y}}_{(i)}\|^2}{(k+1)s^2}
=\frac{\sum_{j=1}^n(\hat y_j-\hat y_{j(i)})^2}{(k+1)s^2} .
$$

**Theorem 4.14** (Cook's distance from one fit; FREES §5.4.2, §5.10.2). Under A4, with $h_{ii}<1$ and $s>0$,

$$
D_i=\frac{(\hat{\boldsymbol{\beta}}-\hat{\boldsymbol{\beta}}_{(i)})'\mathbf{X}'\mathbf{X}(\hat{\boldsymbol{\beta}}-\hat{\boldsymbol{\beta}}_{(i)})}{(k+1)s^2}
=\frac{e_i^2\,h_{ii}}{(k+1)s^2(1-h_{ii})^2}
=\frac{r_i^2}{k+1}\,\frac{h_{ii}}{1-h_{ii}} , \tag{4}
$$

and $D_i\leqslant(n-k-1)h_{ii}/\{(k+1)(1-h_{ii})\}$.

**Proof.** $\hat{\mathbf{Y}}-\hat{\mathbf{Y}}_{(i)}=\mathbf{X}(\hat{\boldsymbol{\beta}}-\hat{\boldsymbol{\beta}}_{(i)})$, whose squared length is $(\hat{\boldsymbol{\beta}}-\hat{\boldsymbol{\beta}}_{(i)})'\mathbf{X}'\mathbf{X}(\hat{\boldsymbol{\beta}}-\hat{\boldsymbol{\beta}}_{(i)})$. By Proposition 4.3(iii), $\hat{\boldsymbol{\beta}}-\hat{\boldsymbol{\beta}}_{(i)}=(\mathbf{X}'\mathbf{X})^{-1}\mathbf{x}_i\,e_i/(1-h_{ii})$, so this equals

$$
\frac{e_i^2}{(1-h_{ii})^2}\,\mathbf{x}_i'(\mathbf{X}'\mathbf{X})^{-1}\mathbf{X}'\mathbf{X}(\mathbf{X}'\mathbf{X})^{-1}\mathbf{x}_i
=\frac{e_i^2\,h_{ii}}{(1-h_{ii})^2} .
$$

Substituting $e_i^2=r_i^2s^2(1-h_{ii})$ gives the last form, and $r_i^2\leqslant n-k-1$ (Theorem 4.5(i)) the bound. ∎

**Remark** (Reading Cook's distance; FREES §5.4.2). $D_i$ is an outlier measure $r_i^2$ times a leverage measure $h_{ii}/(1-h_{ii})$, which increases from $1/(n-1)$ at $h_{ii}=1/n$ to $\infty$ as $h_{ii}\to1$. FREES's typical size substitutes $r_i^2=1$ and $h_{ii}=(k+1)/n$, giving $1/(n-k-1)$, about $1/n$. FREES also compares $D_i$ with $F_{k+1,n-k-1}$. By Unit 3's theorem **the general linear hypothesis** with $\mathbf{R}=\mathbf{I}_{k+1}$, the statistic for $H_0\colon\boldsymbol{\beta}=\boldsymbol{\beta}_0$ is $(\hat{\boldsymbol{\beta}}-\boldsymbol{\beta}_0)'\mathbf{X}'\mathbf{X}(\hat{\boldsymbol{\beta}}-\boldsymbol{\beta}_0)/\{(k+1)s^2\}$, which has the $F_{k+1,n-k-1}$ law at the true $\boldsymbol{\beta}$ under A1–A5, and $D_i$ is its value at $\boldsymbol{\beta}_0=\hat{\boldsymbol{\beta}}_{(i)}$: $D_i$ exceeds the $1-\alpha$ quantile iff $\hat{\boldsymbol{\beta}}_{(i)}$ lies outside the $100(1-\alpha)\%$ confidence region $\{\boldsymbol{\beta}_0\colon F(\boldsymbol{\beta}_0)\leqslant F_{k+1,n-k-1,1-\alpha}\}$ for $\boldsymbol{\beta}$. $D_i$ is bounded (Theorem 4.14), so it does not have an $F$ law; the quantile is a yardstick.

**Example 4.5** (Outliers and high leverage points, continued; FREES §5.4.2). FREES Table 5.6, with Cook's distance computed from the definition by refitting and from Theorem 4.14:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
tab56 <- sapply(1:3, function(code) {
  d <- subset(ol, CODES %in% c(0, code)); m <- lm(Y ~ X, data = d); i <- nrow(d)
  fit_without <- model.matrix(m) %*% coef(lm(Y ~ X, data = d[-i, ]))
  c(r = rstandard(m)[[i]], h = hatvalues(m)[[i]],
    D_refit = sum((fitted(m) - fit_without)^2) / (2 * summary(m)$sigma^2),
    D_formula = rstandard(m)[[i]]^2 / 2 * hatvalues(m)[[i]] / (1 - hatvalues(m)[[i]]),
    D_R = cooks.distance(m)[[i]])
})
colnames(tab56) <- c("A", "B", "C"); round(tab56, 4)
c(high_leverage_cut = 3 * 2 / 20, typical_D = 1 / 20, F_2_18_95 = qf(0.95, 2, 18))

A and C are outliers ($|r|>2$); B and C are high leverage points, both with $h=0.550>3(k+1)/n=0.3$, equal because they share $x=9.5$. Cook's distance flags all three against FREES's typical value $1/n=0.05$, and only C, with $D=9.83$, exceeds the $F_{2,18}$ quantile $3.555$, so only deleting C moves $\hat{\boldsymbol{\beta}}$ outside the $95\%$ confidence region.

**Definition 4.15** (DFFITS and DFBETAS (beyond the syllabus)). For $0<h_{ii}<1$ and $s_{(i)}>0$,

$$
\mathrm{DFFITS}_i=\frac{\hat y_i-\hat y_{i(i)}}{s_{(i)}\sqrt{h_{ii}}},\qquad
\mathrm{DFBETAS}_{ij}=\frac{\hat\beta_j-\hat\beta_{j(i)}}{s_{(i)}\sqrt{[(\mathbf{X}'\mathbf{X})^{-1}]_{jj}}},
\quad j=0,\ldots,k,
$$

where $\hat y_{i(i)}=\mathbf{x}_i'\hat{\boldsymbol{\beta}}_{(i)}$ and $\hat\beta_{j(i)}$ is the $j$-th entry of $\hat{\boldsymbol{\beta}}_{(i)}$.

**Proposition 4.16** (DFFITS and DFBETAS from one fit (beyond the syllabus)). With an intercept, under A4, $h_{ii}<1$, $s>0$ and $s_{(i)}>0$,

$$
\mathrm{DFFITS}_i=t_i\sqrt{\frac{h_{ii}}{1-h_{ii}}},\qquad
D_i=\frac{s_{(i)}^2}{(k+1)s^2}\,\mathrm{DFFITS}_i^2 ,
$$

$$
\mathrm{DFBETAS}_{ij}=\frac{[(\mathbf{X}'\mathbf{X})^{-1}\mathbf{x}_i]_j\,e_i}{(1-h_{ii})\,s_{(i)}\sqrt{[(\mathbf{X}'\mathbf{X})^{-1}]_{jj}}} .
$$

**Proof.** With an intercept $h_{ii}\geqslant1/n>0$, so the divisions are defined. By Proposition 4.3(iii), $\hat y_i-\hat y_{i(i)}=\mathbf{x}_i'(\mathbf{X}'\mathbf{X})^{-1}\mathbf{x}_i\,e_i/(1-h_{ii})=h_{ii}e_i/(1-h_{ii})$; dividing by $s_{(i)}\sqrt{h_{ii}}$ gives $e_i\sqrt{h_{ii}}/\{s_{(i)}(1-h_{ii})\}=t_i\sqrt{h_{ii}/(1-h_{ii})}$. The $j$-th entry of $\hat{\boldsymbol{\beta}}-\hat{\boldsymbol{\beta}}_{(i)}=(\mathbf{X}'\mathbf{X})^{-1}\mathbf{x}_ie_i/(1-h_{ii})$ gives DFBETAS. Finally $r_i=t_is_{(i)}/s$, so Theorem 4.14 gives $D_i=t_i^2s_{(i)}^2h_{ii}/\{(k+1)s^2(1-h_{ii})\}$. ∎

**Example 4.6** (Stock market liquidity, continued: IBM; FREES §5.5.2 and chapter 5 script). The regression of `VOLUME` on `PRICE`, `SHARE` and `VALUE`, whose diagnostics close FREES's liquidity analysis:

In [ ]:
mv <- lm(VOLUME ~ PRICE + SHARE + VALUE, data = liq); k <- 3
infl <- cbind(h = hatvalues(mv), t = rstudent(mv), D = cooks.distance(mv))
round(infl[order(-infl[, "D"])[1:4], ], 4)
c(high_leverage_cut = 3 * (k + 1) / n)
liq[c(60, 122), c("COMPANY", "VOLUME", "PRICE", "SHARE", "VALUE")]
Xt <- as.matrix(liq[c("PRICE", "SHARE", "VALUE")]); S <- cov(Xt)
round(c((Xt[60, ] - colMeans(Xt)) / sqrt(diag(S)), Delta2 = mahalanobis(Xt[60, ], colMeans(Xt), S),
        h_from_Delta2 = 1 / n + mahalanobis(Xt[60, ], colMeans(Xt), S) / (n - 1),
        h_over_1_minus_h = infl[60, "h"] / (1 - infl[60, "h"])), 4)

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.5)
op <- par(mar = c(3.2, 3.4, 0.6, 0.6), mgp = c(1.9, 0.6, 0))
plot(infl[, "h"], infl[, "t"], cex = 0.4 + 3 * sqrt(infl[, "D"] / max(infl[, "D"])),
     xlab = "leverage", ylab = "studentised residual", xlim = c(0, 0.95))
abline(v = 3 * (k + 1) / n, lty = 2); abline(h = c(-2, 2), lty = 3)
text(infl[c(60, 122), "h"], infl[c(60, 122), "t"], c("IBM", "Exxon"), pos = c(2, 4), cex = 0.8)
par(op)

*Figure 4.5.* Studentised residuals against leverage, circle areas growing with Cook's distance; dashed line at $3(k+1)/n$, dotted lines at $\pm2$.

IBM has leverage $0.876$, nine times the cutoff $3(k+1)/n=0.098$: its price, shares and value are $3.9$, $4.5$ and $8.7$ standard deviations above the means, and its squared Mahalanobis distance $105.8$ reproduces $h$ through (3). Its studentised residual is only $1.22$, yet its Cook's distance $2.60$ is the largest, through the leverage factor $h/(1-h)=7.0$. Exxon combines leverage $0.337$ with $t=-3.73$. Refitting without IBM:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
mv60 <- update(mv, data = liq[-60, ])
round(rbind(all = coef(summary(mv))[, 3], without_IBM = coef(summary(mv60))[, 3]), 3)
round(rbind(all = coef(mv), without_IBM = coef(mv60)), 5)
round(rbind(all = car::vif(mv), without_IBM = car::vif(mv60)), 3)
c(R2_all = summary(mv)$r.squared, R2_without_IBM = summary(mv60)$r.squared)

The coefficient of `VALUE` falls from $0.313$ ($t=1.94$) to $-0.125$ ($t=-0.32$) and $R^2$ from $0.610$ to $0.524$: one company in $123$ carries the evidence for `VALUE`.

## 4 Collinearity

> **Reading.** FREES §5.5, §5.10.3 · ISLR §3.3.3 (collinearity) · CLM §6. SRM learning outcomes 2(d), 2(e).

### 4.1 What collinearity does

**Example 4.7** (Perfectly correlated explanatory variables; FREES §5.5.1). Joe fits $\hat y=-87+x_1+18x_2$ and Jane fits $\hat y=-7+9x_1+2x_2$ to four observations with $x_2=5+x_1/2$:

In [ ]:
jf <- data.frame(y = c(23, 83, 63, 103), x1 = c(2, 8, 6, 10), x2 = c(6, 9, 8, 10))
Xj <- cbind(1, jf$x1, jf$x2)
cbind(y = jf$y, joe = drop(Xj %*% c(-87, 1, 18)), jane = drop(Xj %*% c(-7, 9, 2)),
      null_direction = drop(Xj %*% c(10, 1, -2)))
c(rank = qr(Xj)$rank); coef(lm(y ~ x1 + x2, data = jf))

Both fit every observation exactly, and so does R's $(3,10,0)$, which drops $x_2$.

**Proposition 4.17** (Exact collinearity; FREES §5.5.1). If $\operatorname{rank}(\mathbf{X})<k+1$, the solutions of the normal equations $\mathbf{X}'\mathbf{X}\boldsymbol{\beta}=\mathbf{X}'\mathbf{Y}$ form the set $\hat{\boldsymbol{\beta}}_0+\mathcal{N}(\mathbf{X})$, for any one solution $\hat{\boldsymbol{\beta}}_0$, which is infinite; all of them give the same fitted values, the orthogonal projection of $\mathbf{Y}$ onto $\mathcal{C}(\mathbf{X})$, hence the same residuals, $\mathrm{SSE}$ and $R^2$.

**Proof.** By the prerequisites' theorem **orthogonal projection** there is a unique $\hat{\mathbf{Y}}\in\mathcal{C}(\mathbf{X})$ with $\mathbf{Y}-\hat{\mathbf{Y}}\perp\mathcal{C}(\mathbf{X})$; writing $\hat{\mathbf{Y}}=\mathbf{X}\hat{\boldsymbol{\beta}}_0$, this says $\mathbf{X}'(\mathbf{Y}-\mathbf{X}\hat{\boldsymbol{\beta}}_0)=\mathbf{0}$, so $\hat{\boldsymbol{\beta}}_0$ solves the normal equations. Any solution $\boldsymbol{\beta}$ has $\mathbf{X}\boldsymbol{\beta}\in\mathcal{C}(\mathbf{X})$ and $\mathbf{Y}-\mathbf{X}\boldsymbol{\beta}\perp\mathcal{C}(\mathbf{X})$, so $\mathbf{X}\boldsymbol{\beta}=\hat{\mathbf{Y}}$ by uniqueness and $\boldsymbol{\beta}-\hat{\boldsymbol{\beta}}_0\in\mathcal{N}(\mathbf{X})$; conversely every $\hat{\boldsymbol{\beta}}_0+\mathbf{v}$ with $\mathbf{X}\mathbf{v}=\mathbf{0}$ solves them. By the prerequisites' rank–nullity theorem, $\dim\mathcal{N}(\mathbf{X})=k+1-\operatorname{rank}(\mathbf{X})\geqslant1$. ∎

**Remark** (FREES §5.5.1). In Example 4.7, $\mathcal{N}(\mathbf{X})$ is spanned by $(10,1,-2)'$, since $10+x_1-2x_2=0$ on the data; Jane's coefficients are Joe's plus $8(10,1,-2)'$ and R's are Joe's plus $9(10,1,-2)'$. The fit is determined and the coefficients are not.

**Proposition 4.18** (What collinearity does and does not affect; FREES §5.5.1–5.5.2). Suppose the model has an intercept and A1–A4, and let $R_j^2$ and $s_{x_j}$ be as in Unit 3's corollary **the standard error through the other explanatory variables**.

(i) $\operatorname{\mathsf{E}}[s^2]=\sigma^2$ and $\operatorname{var}(\hat y_i)=\sigma^2h_{ii}\leqslant\sigma^2$ for every $i$, whatever the correlations among the explanatory variables.

(ii) $\operatorname{var}(\hat\beta_j)=\sigma^2\mathrm{VIF}_j/\{(n-1)s_{x_j}^2\}$, which for fixed $s_{x_j}$ grows without bound as $R_j^2\to1$.

(iii) The slope block of $(\mathbf{X}'\mathbf{X})^{-1}$ is $(\tilde{\mathbf{X}}_c'\tilde{\mathbf{X}}_c)^{-1}$; for $k=2$, $\operatorname{cor}(\hat\beta_1,\hat\beta_2)=-r_{12}$, the negative of the sample correlation of $x_1$ and $x_2$.

**Proof.** **(i)** $\operatorname{\mathsf{E}}[s^2]=\sigma^2$ is Unit 2's theorem **unbiasedness of the mean square error**. $\hat{\mathbf{Y}}=\mathbf{H}\mathbf{Y}$, so $\operatorname{var}(\hat{\mathbf{Y}})=\mathbf{H}(\sigma^2\mathbf{I})\mathbf{H}=\sigma^2\mathbf{H}$, whose diagonal entries are $\sigma^2h_{ii}\leqslant\sigma^2$ by Unit 2's theorem **properties of leverage**.

**(ii)** Unit 3's corollary cited in the statement.

**(iii)** In the proof of Proposition 4.12, $(\mathbf{X}'\mathbf{X})^{-1}=\mathbf{A}\operatorname{diag}(1/n,\mathbf{C})\mathbf{A}'$ with $\mathbf{C}=(\tilde{\mathbf{X}}_c'\tilde{\mathbf{X}}_c)^{-1}$ and $\mathbf{A}=\begin{pmatrix}1&-\bar{\mathbf{x}}'\\\mathbf{0}&\mathbf{I}_k\end{pmatrix}$; multiplying out,

$$
(\mathbf{X}'\mathbf{X})^{-1}=\begin{pmatrix}1/n+\bar{\mathbf{x}}'\mathbf{C}\bar{\mathbf{x}}&-\bar{\mathbf{x}}'\mathbf{C}\\-\mathbf{C}\bar{\mathbf{x}}&\mathbf{C}\end{pmatrix} .
$$

For $k=2$, with $s_1,s_2$ the standard deviations of $x_1,x_2$ and $r=r_{12}$,

$$
\tilde{\mathbf{X}}_c'\tilde{\mathbf{X}}_c=(n-1)\begin{pmatrix}s_1^2&rs_1s_2\\rs_1s_2&s_2^2\end{pmatrix},\qquad
\mathbf{C}=\frac{1}{(n-1)s_1^2s_2^2(1-r^2)}\begin{pmatrix}s_2^2&-rs_1s_2\\-rs_1s_2&s_1^2\end{pmatrix},
$$

$|r|<1$ by A4. By Unit 2's theorem **variance–covariance matrix**, $\operatorname{cor}(\hat\beta_1,\hat\beta_2)=C_{12}/\sqrt{C_{11}C_{22}}=-rs_1s_2/(s_1s_2)=-r$. ∎

**Remark** (Collinearity facts; FREES §5.5.1, ISLR §3.3.3). Part (i) is FREES's first two facts: collinearity prevents neither a good fit nor reliable estimates of $\sigma^2$ and of the overall fit. Predictions are another matter: by Proposition 4.12 the variance of the fitted mean at a new $\mathbf{x}_0$ is $\sigma^2\{1/n+\Delta_0^2/(n-1)\}$, small where $\mathbf{x}_0$ follows the pattern of the data and large where it breaks a near-linear relation, even with each coordinate inside its observed range. Part (ii) is the third fact and ISLR's point: large standard errors, small $t$ ratios, and tests of $H_0\colon\beta_j=0$ that lose power.

**Proposition 4.19** (The residual sum of squares around its minimum; ISLR §3.3.3). Suppose the model has an intercept and A4, let $\hat{\boldsymbol{\beta}}_s$ be the least squares slopes and $\tilde{\mathbf{X}},\tilde{\mathbf{X}}_c$ as in Proposition 4.12. For every $\boldsymbol{\beta}_s\in\mathbb{R}^k$,

$$
\min_{\beta_0\in\mathbb{R}}\|\mathbf{Y}-\beta_0\mathbf{1}-\tilde{\mathbf{X}}\boldsymbol{\beta}_s\|^2
=\mathrm{SSE}+(\boldsymbol{\beta}_s-\hat{\boldsymbol{\beta}}_s)'\tilde{\mathbf{X}}_c'\tilde{\mathbf{X}}_c(\boldsymbol{\beta}_s-\hat{\boldsymbol{\beta}}_s) .
$$

The contours in $\boldsymbol{\beta}_s$ are therefore ellipsoids centred at $\hat{\boldsymbol{\beta}}_s$, with axes along the eigenvectors of $\tilde{\mathbf{X}}_c'\tilde{\mathbf{X}}_c$ and semi-axes proportional to the reciprocal square roots of its eigenvalues. For $k=2$ variables with unit standard deviations and correlation $r_{12}$, the eigenvalues are $(n-1)(1\pm r_{12})$ and the longest axis is $\sqrt{(1+|r_{12}|)/(1-|r_{12}|)}$ times the shortest.

**Proof.** Put $\alpha=\beta_0+\bar{\mathbf{x}}'\boldsymbol{\beta}_s$, so that $\beta_0\mathbf{1}+\tilde{\mathbf{X}}\boldsymbol{\beta}_s=\alpha\mathbf{1}+\tilde{\mathbf{X}}_c\boldsymbol{\beta}_s$, a bijection $(\beta_0,\boldsymbol{\beta}_s)\leftrightarrow(\alpha,\boldsymbol{\beta}_s)$ that keeps the slopes. By Unit 3's proposition **codings are reparametrisations**, the least squares fit in the design $[\mathbf{1}\ \tilde{\mathbf{X}}_c]=\mathbf{X}\mathbf{A}$ has the same fitted values and coefficients $\mathbf{A}^{-1}\hat{\boldsymbol{\beta}}=(\hat\alpha,\hat{\boldsymbol{\beta}}_s')'$ with $\hat\alpha=\hat\beta_0+\bar{\mathbf{x}}'\hat{\boldsymbol{\beta}}_s$, so $\mathbf{Y}=\hat\alpha\mathbf{1}+\tilde{\mathbf{X}}_c\hat{\boldsymbol{\beta}}_s+\mathbf e$ and

$$
\mathbf{Y}-\alpha\mathbf{1}-\tilde{\mathbf{X}}_c\boldsymbol{\beta}_s=\mathbf e+(\hat\alpha-\alpha)\mathbf{1}
+\tilde{\mathbf{X}}_c(\hat{\boldsymbol{\beta}}_s-\boldsymbol{\beta}_s) .
$$

The three terms are mutually orthogonal: $\mathbf e\perp\mathcal{C}(\mathbf{X})$, and $\mathbf{1}'\tilde{\mathbf{X}}_c=\mathbf{0}'$. By the prerequisites' Pythagoras theorem the squared length is $\mathrm{SSE}+n(\hat\alpha-\alpha)^2+(\hat{\boldsymbol{\beta}}_s-\boldsymbol{\beta}_s)'\tilde{\mathbf{X}}_c'\tilde{\mathbf{X}}_c(\hat{\boldsymbol{\beta}}_s-\boldsymbol{\beta}_s)$, minimised over $\alpha$ at $\alpha=\hat\alpha$. By the prerequisites' spectral theorem, $\tilde{\mathbf{X}}_c'\tilde{\mathbf{X}}_c=\mathbf{P}\boldsymbol{\Lambda}\mathbf{P}'$ with $\mathbf{P}$ orthogonal; in the coordinates $\mathbf{w}=\mathbf{P}'(\boldsymbol{\beta}_s-\hat{\boldsymbol{\beta}}_s)$ the contour at level $\mathrm{SSE}+c$ is $\sum_j\lambda_jw_j^2=c$, with semi-axes $\sqrt{c/\lambda_j}$ along the columns of $\mathbf{P}$. For $k=2$ unit-variance variables, $\tilde{\mathbf{X}}_c'\tilde{\mathbf{X}}_c=(n-1)\begin{pmatrix}1&r_{12}\\r_{12}&1\end{pmatrix}$, with eigenvectors $(1,1)'/\sqrt2$ and $(1,-1)'/\sqrt2$ and eigenvalues $(n-1)(1\pm r_{12})$; the axes are in the ratio $\sqrt{(1+|r_{12}|)/(1-|r_{12}|)}$. ∎

**Example 4.8** (Credit; ISLR §3.3.3). ISLR Table 3.11: `Balance` on `Age` and `Limit`, and on `Rating` and `Limit`.

In [ ]:
cm1 <- lm(Balance ~ Age + Limit, data = Credit); cm2 <- lm(Balance ~ Rating + Limit, data = Credit)
round(coef(summary(cm1)), 4); round(coef(summary(cm2)), 4)
c(se_ratio_Limit = coef(summary(cm2))["Limit", 2] / coef(summary(cm1))["Limit", 2],
  r_Age_Limit = cor(Credit$Age, Credit$Limit), r_Rating_Limit = cor(Credit$Rating, Credit$Limit))
axis_ratio <- function(r) sqrt((1 + abs(r)) / (1 - abs(r)))
c(Age_Limit = axis_ratio(cor(Credit$Age, Credit$Limit)),
  Rating_Limit = axis_ratio(cor(Credit$Rating, Credit$Limit)))
V2 <- vcov(cm2); c(cor_b_Rating_b_Limit = V2[2, 3] / sqrt(V2[2, 2] * V2[3, 3]))

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.6)
rss_grid <- function(m, v) {
  b <- coef(m)[v]; se <- sqrt(diag(vcov(m)))[v]
  Xc <- scale(as.matrix(Credit[v]), scale = FALSE); G <- crossprod(Xc)
  g1 <- seq(b[1] - 4 * se[1], b[1] + 4 * se[1], length = 121)
  g2 <- seq(b[2] - 4 * se[2], b[2] + 4 * se[2], length = 121)
  z <- outer(g1, g2, Vectorize(function(u, w) { d <- c(u, w) - b; drop(t(d) %*% G %*% d) }))
  lev <- sum(residuals(m)^2) + 2 * summary(m)$sigma^2 * qf(c(0.5, 0.9, 0.99, 0.999), 2, nrow(Credit) - 3)
  list(g1 = g1, g2 = g2, z = (sum(residuals(m)^2) + z) / 1e6, b = b, lev = lev / 1e6)
}
op <- par(mfrow = c(1, 2), mar = c(3.2, 3.4, 0.6, 0.6), mgp = c(1.9, 0.6, 0))
for (spec in list(list(cm1, c("Limit", "Age")), list(cm2, c("Limit", "Rating")))) {
  g <- rss_grid(spec[[1]], spec[[2]])
  contour(g$g1, g$g2, g$z, levels = g$lev, drawlabels = FALSE,
          xlab = paste("beta", spec[[2]][1]), ylab = paste("beta", spec[[2]][2]))
  points(g$b[1], g$b[2], pch = 16)
}
par(op)

*Figure 4.6.* Contours of the residual sum of squares, the intercept minimised out, over $\pm4$ standard errors of the two slopes, at $\mathrm{SSE}+2s^2F_{2,n-3,1-\alpha}$ for $\alpha=0.5,0.1,0.01,0.001$ (ISLR Figure 3.15).

`Limit` and `Rating` have correlation $0.997$, and in their regression the standard error of the `Limit` coefficient is $12.7$ times that in the regression with `Age`, so `Limit` loses its significance ($p=0.70$). By Proposition 4.19 the standardised ellipses have axis ratio $25.3$ against $1.11$ with `Age`: the long valley of the right panel, along which the two coefficients trade off with correlation $-0.997$ (Proposition 4.18(iii)). The contours drawn are the boundaries of the joint confidence regions for the two slopes: their quadratic form $(\boldsymbol{\beta}_s-\hat{\boldsymbol{\beta}}_s)'\tilde{\mathbf{X}}_c'\tilde{\mathbf{X}}_c(\boldsymbol{\beta}_s-\hat{\boldsymbol{\beta}}_s)$ is, by Proposition 4.18(iii), the numerator of Unit 3's general linear hypothesis statistic for the two slopes, with $q=2$.

### 4.2 Variance inflation factors

**Proposition 4.20** (Variance inflation factors; FREES §5.5.2, ISLR §3.3.3). Suppose the model has an intercept, $k\geqslant2$ and A4; part (ii) also assumes A1–A3.

(i) $\mathrm{VIF}_j\geqslant1$, with equality iff $x_j$ has sample correlation $0$ with every other explanatory variable.

(ii) $\mathrm{VIF}_j$ is the variance of $\hat\beta_j$ in the full regression divided by the variance of the slope of the simple regression of $y$ on $x_j$ alone, ISLR's definition.

(iii) The **tolerance** $1/\mathrm{VIF}_j$ equals $1-R_j^2$; $\mathrm{VIF}_j>10$ iff $R_j^2>0.9$, and $\mathrm{VIF}_j>5$ iff $R_j^2>0.8$.

(iv) Centring and rescaling the explanatory variables leave every $\mathrm{VIF}_j$ unchanged.

**Proof.** **(i)** Let $\mathbf{X}_{(j)}$ be $\mathbf{X}$ without $\mathbf{x}_j$ and $\mathbf{Z}_c$ its $k-1$ non-constant columns centred. By (2) for the design $\mathbf{X}_{(j)}$, the regression of $\mathbf{x}_j$ on the other variables has regression sum of squares $\|\mathbf{Z}_c(\mathbf{Z}_c'\mathbf{Z}_c)^{-1}\mathbf{Z}_c'\mathbf{x}_j\|^2\geqslant0$, so $R_j^2\geqslant0$, and $R_j^2<1$ by Unit 3's corollary **the standard error through the other explanatory variables**. Hence $\mathrm{VIF}_j=1/(1-R_j^2)\geqslant1$, with equality iff $\mathbf{Z}_c(\mathbf{Z}_c'\mathbf{Z}_c)^{-1}\mathbf{Z}_c'\mathbf{x}_j=\mathbf{0}$, which, multiplying by $\mathbf{Z}_c'$, holds iff $\mathbf{Z}_c'\mathbf{x}_j=\mathbf{0}$. The entries of $\mathbf{Z}_c'\mathbf{x}_j$ are $(n-1)$ times the sample covariances of $x_j$ with the other variables, $\mathbf{Z}_c$ having columns that sum to zero.

**(ii)** The slope of the simple regression is $\sum_iw_iy_i$ with $\sum_iw_i^2=1/\{(n-1)s_{x_j}^2\}$ (Unit 2's proposition **coefficients are weighted sums of the responses**), so under A3 its variance is $\sigma^2/\{(n-1)s_{x_j}^2\}$ by the prerequisites' proposition **variance under a linear map**, whether or not its mean is correct; divide Unit 3's formula $\operatorname{var}(\hat\beta_j)=\sigma^2\mathrm{VIF}_j/\{(n-1)s_{x_j}^2\}$ by it.

**(iii)** $1/(1-R_j^2)>10$ iff $1-R_j^2<0.1$; likewise for $5$.

**(iv)** Replacing each $x_l$ by $a_lx_l+c_l$ with $a_l\neq0$ replaces $\mathbf{X}_{(j)}$ by $\mathbf{X}_{(j)}\mathbf{B}$ for an invertible $\mathbf{B}$, which leaves its column space and hat matrix unchanged, and replaces $\mathbf{x}_j$ by $a_j\mathbf{x}_j+c_j\mathbf{1}$. Since $\mathbf{1}\in\mathcal{C}(\mathbf{X}_{(j)})$, the new residuals are $a_j$ times the old and the new total sum of squares is $a_j^2$ times the old, so $R_j^2=1-\mathrm{SSE}/\mathrm{SST}$ is unchanged. ∎

**Example 4.9** (Credit and liquidity; ISLR §3.3.3, FREES §5.5.2). ISLR's three-variable `Credit` regression and FREES Table 5.7, with FREES's equation (5.5) checked for each coefficient:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
cm3 <- lm(Balance ~ Age + Rating + Limit, data = Credit)
round(car::vif(cm3), 2)
c(R2_all_three = summary(cm3)$r.squared, R2_Age_Limit = summary(cm1)$r.squared)
cat(sprintf("cor(PRICE*SHARE/1000, VALUE) = %.7f, max gap = %.2f\n",
    cor(liq$PRICE * liq$SHARE / 1000, liq$VALUE), max(abs(liq$PRICE * liq$SHARE / 1000 - liq$VALUE))))
round(cbind(s_x = sapply(liq[c("PRICE", "SHARE", "VALUE")], sd), coef(summary(mv))[-1, 1:3],
            VIF = car::vif(mv),
            se_from_VIF = summary(mv)$sigma * sqrt(car::vif(mv)) /
              (sapply(liq[c("PRICE", "SHARE", "VALUE")], sd) * sqrt(n - 1))), 4)
c(R2 = summary(mv)$r.squared, s = summary(mv)$sigma,
  VIF_PRICE_aux = 1 / (1 - summary(lm(PRICE ~ SHARE + VALUE, data = liq))$r.squared))

The `Credit` factors are $1.01$, $160.67$ and $160.59$: `Rating` and `Limit` each have $R_j^2>0.99$ on the others. For the liquidity regression, $R^2=61\%$, $s=6.72$, and the factors $1.5$, $3.8$ and $4.7$ are all below $10$, although `VALUE` is $\texttt{PRICE}\times\texttt{SHARE}/1000$ to within $0.46$ billion (correlation $0.999996$): the relation is multiplicative, not linear, and the variance inflation factor measures only linear dependence.

**Proposition 4.21** (Centring a variable and its square; FREES §5.5.2). Let $x_1,\ldots,x_n$ take at least three distinct values and let $c_i=x_i-\bar x$. The regressions of $y$ on $(x,x^2)$ and on $(c,c^2)$ have the same fitted values, and the sample correlation of $c$ and $c^2$ is $0$ iff $\sum_ic_i^3=0$, which holds when the $x_i$ are symmetric about $\bar x$.

**Proof.** $c=x-\bar x\mathbf{1}$ and $c^2=x^2-2\bar x\,x+\bar x^2\mathbf{1}$ entrywise, so $[\mathbf{1}\ c\ c^2]=[\mathbf{1}\ x\ x^2]\mathbf{A}$ with $\mathbf{A}$ upper triangular with unit diagonal; three distinct values make $[\mathbf{1}\ x\ x^2]$ of full rank (a non-zero quadratic has at most two roots), and Unit 3's proposition **codings are reparametrisations** gives the same fitted values. If the $c_i^2$ were all equal to some $d$, every $c_i$ would be $\sqrt d$ or $-\sqrt d$ and the $x_i$ would take at most two values; so the $c_i^2$ are not all equal. With $\sum_ic_i=0$, the sample covariance of $c$ and $c^2$ is $\sum_ic_i(c_i^2-\overline{c^2})/(n-1)=\sum_ic_i^3/(n-1)$. Symmetry about $\bar x$ pairs each $c_i$ with $-c_i$ and makes $\sum_ic_i^3=0$. ∎

**Example 4.10** (Centring horsepower; FREES §5.5.2).

In [ ]:
hp <- Auto$horsepower; hc <- hp - mean(hp)
q_raw <- lm(Auto$mpg ~ hp + I(hp^2)); q_cen <- lm(Auto$mpg ~ hc + I(hc^2))
range(hp)
round(c(cor_raw = cor(hp, hp^2), VIF_raw = 1 / (1 - cor(hp, hp^2)^2),
  cor_centred = cor(hc, hc^2), VIF_centred = 1 / (1 - cor(hc, hc^2)^2)), 3)
signif(c(sum_c_cubed_over_n = mean(hc^3), fitted_diff = max(abs(fitted(q_raw) - fitted(q_cen)))), 3)

Over the sampled range, $46$ to $230$, `horsepower` and its square are nearly linearly related, with correlation $0.983$ and variance inflation factor $29.3$ (FREES Figure 5.5). Centring lowers the correlation to $0.663$ and the factor to $1.78$ without changing the fit; not to $0$, because the distribution of `horsepower` is skewed to the right and $\sum_ic_i^3>0$.

**Remark** (Remedies; FREES §5.5.2, ISLR §3.3.3). Rescaling variables of very different magnitudes is a computational remedy only: by Proposition 4.20(iv) it changes no variance inflation factor, and by Unit 3's proposition **codings are reparametrisations** no fit. The statistical remedies are to drop one of the collinear variables or to combine them. In Example 4.9, dropping `Rating` lowers $R^2$ from $0.7536$ to $0.7498$.

In [ ]:
Credit$worth <- drop(scale(Credit$Limit) + scale(Credit$Rating)) / 2
cm4 <- lm(Balance ~ Age + worth, data = Credit)
c(R2 = summary(cm4)$r.squared, car::vif(cm4))

The average of the standardised `Limit` and `Rating`, a creditworthiness score, keeps $R^2=0.7529$ with both variance inflation factors at $1.01$.

### 4.3 Collinearity and leverage

**Proposition 4.22** (One point and collinearity; FREES §5.5.3). Let $(u_1,v_1),\ldots,(u_m,v_m)$ be fixed, $m\geqslant2$, add the point $(ca,cb)$ with $a\neq0\neq b$, and let $r(c)$ be the sample correlation of the $m+1$ points.

(i) $r(c)\to\operatorname{sign}(ab)$ as $c\to\infty$, so the variance inflation factor $1/\{1-r(c)^2\}$ of either variable in a regression on both grows without bound.

(ii) If the $m$ points are not all equal and lie on one line, and $(ca,cb)$ is off that line, then $|r(c)|<1$.

**Proof.** **(i)** With $N=m+1$, $U=\sum_{i\leqslant m}u_i$, $V=\sum_{i\leqslant m}v_i$ and $P=\sum_{i\leqslant m}u_iv_i$, the centred cross product of the $N$ points is

$$
S_{uv}(c)=P+c^2ab-\frac{(U+ca)(V+cb)}{N}
=c^2ab\Bigl(1-\frac1N\Bigr)-c\,\frac{aV+bU}{N}+P-\frac{UV}{N} ,
$$

and likewise $S_{uu}(c)=c^2a^2(1-1/N)+O(c)$, $S_{vv}(c)=c^2b^2(1-1/N)+O(c)$. Dividing numerator and denominator of $r(c)=S_{uv}/\sqrt{S_{uu}S_{vv}}$ by $c^2$, $r(c)\to ab/\sqrt{a^2b^2}=\operatorname{sign}(ab)$. For $k=2$, $R_1^2=R_2^2=r^2$, Unit 2's identity $R^2=r^2$ for one explanatory variable.

**(ii)** By the prerequisites' Cauchy–Schwarz theorem, $|r|=1$ iff the centred vectors $(u_i-\bar u)$ and $(v_i-\bar v)$ are proportional, that is iff all $N$ points lie on one line. The only line through the $m$ points, which are not all equal, is the given one, and $(ca,cb)$ is off it. ∎

**Example 4.11** (Masking and inducing; FREES §5.5.3). Thirty nearly collinear points plus one point off their line, and thirty uncorrelated points plus one point moved out along the diagonal (FREES Figures 5.6 and 5.7):

In [ ]:
set.seed(11); x1 <- rnorm(30); x2 <- x1 + rnorm(30, sd = 0.15)
vif2 <- function(a, b) 1 / (1 - cor(a, b)^2)
c(r_30 = cor(x1, x2), VIF_30 = vif2(x1, x2),
  r_31 = cor(c(x1, 2.5), c(x2, -2.5)), VIF_31 = vif2(c(x1, 2.5), c(x2, -2.5)))
z1 <- rnorm(30); z2 <- rnorm(30)
round(sapply(c(0, 5, 20, 100), function(cc)
  c(c = cc, r = cor(c(z1, cc), c(z2, cc)), VIF = vif2(c(z1, cc), c(z2, cc)))), 3)

One point off the line of the first thirty cuts the variance inflation factor from $28.8$ to $1.23$; moving one point out along the diagonal of the second cloud drives the correlation from $-0.18$ to $0.996$ at $c=100$, with variance inflation factor $138$, as Proposition 4.22(i) predicts. The same happens in Example 4.6: without IBM the factors of `SHARE` and `VALUE` rise from $3.8$ and $4.7$ to $9.4$ and $10.3$, so the high leverage point was masking the collinearity, and through Unit 3's standard error formula it moved the standard errors too.

### 4.4 Suppressor variables

**Proposition 4.23** ($R^2$ from correlations; FREES §5.5.4). For the regression of a non-constant $y$ on $x_1,\ldots,x_k$ with an intercept, let $\mathbf{r}_y$ be the vector of sample correlations $r(y,x_j)$ and $\mathbf{R}_{xx}$ the correlation matrix of the $x$'s, invertible under A4. Then $R^2=\mathbf{r}_y'\mathbf{R}_{xx}^{-1}\mathbf{r}_y$, and for $k=2$

$$
R^2=\frac{r_{y1}^2+r_{y2}^2-2r_{y1}r_{y2}r_{12}}{1-r_{12}^2} . \tag{5}
$$

**Proof.** By (2), $\mathrm{SSR}=\|(\mathbf{H}-\tfrac1n\mathbf{1}\mathbf{1}')\mathbf{Y}\|^2=\mathbf{Y}'\tilde{\mathbf{X}}_c(\tilde{\mathbf{X}}_c'\tilde{\mathbf{X}}_c)^{-1}\tilde{\mathbf{X}}_c'\mathbf{Y}$. With $\mathbf{D}=\operatorname{diag}(s_{x_1},\ldots,s_{x_k})$ and $s_y$ the standard deviation of $y$, $\tilde{\mathbf{X}}_c'\mathbf{Y}=(n-1)s_y\mathbf{D}\mathbf{r}_y$ and $\tilde{\mathbf{X}}_c'\tilde{\mathbf{X}}_c=(n-1)\mathbf{D}\mathbf{R}_{xx}\mathbf{D}$, so $\mathrm{SSR}=(n-1)s_y^2\mathbf{r}_y'\mathbf{R}_{xx}^{-1}\mathbf{r}_y$; divide by $\mathrm{SST}=(n-1)s_y^2>0$. For $k=2$, $\mathbf{R}_{xx}^{-1}=(1-r_{12}^2)^{-1}\begin{pmatrix}1&-r_{12}\\-r_{12}&1\end{pmatrix}$. ∎

**Example 4.12** (Suppressor variable; FREES §5.5.4). FREES Table 5.8 reports $r_{12}=0.972$, $r_{y1}=0.188$ and $r_{y2}=-0.022$ for $50$ hypothetical observations, and $R^2=80.7\%$ for the regression on both:

In [ ]:
R2k2 <- function(r1, r2, r12) (r1^2 + r2^2 - 2 * r1 * r2 * r12) / (1 - r12^2)
g <- expand.grid(r1 = c(0.1875, 0.1885), r2 = c(-0.0225, -0.0215), r12 = c(0.9715, 0.9725))
c(printed = R2k2(0.188, -0.022, 0.972), min_rounding = min(with(g, R2k2(r1, r2, r12))),
  max_rounding = max(with(g, R2k2(r1, r2, r12))), r_y2_zero = R2k2(0.188, 0, 0.972),
  r_y1_sq = 0.188^2, one_minus_r12_sq = 1 - 0.972^2)
set.seed(5); x1 <- rnorm(50); x2 <- 0.972 * x1 + sqrt(1 - 0.972^2) * rnorm(50)
y <- 3 * (x1 - x2) + 0.3 * rnorm(50)
round(cor(cbind(x1, x2, y)), 3)
c(R2_x1 = summary(lm(y ~ x1))$r.squared, R2_x2 = summary(lm(y ~ x2))$r.squared,
  R2_both = summary(lm(y ~ x1 + x2))$r.squared)

The printed correlations give $R^2=0.794$ by (5), and correlations that round to them give anything from $0.773$ to $0.816$, which contains FREES's $0.807$: the denominator $1-r_{12}^2=0.055$ magnifies rounding. With $r_{y2}=0$ the formula is $r_{y1}^2/(1-r_{12}^2)$, so a variable uncorrelated with $y$ raises $R^2$ from $0.035$ to $0.640$; it **suppresses** the part of $x_1$ unrelated to $y$. The simulated data reproduce the pattern: $R^2$ of $0.2\%$ and $1.0\%$ separately and $71\%$ together.

### 4.5 Orthogonal variables

**Proposition 4.24** (Orthogonal explanatory variables; FREES §5.5.5). Let $\mathbf{X}=[\mathbf{X}_1\ \mathbf{X}_2]$ satisfy A4 with $\mathbf{X}_1'\mathbf{X}_2=\mathbf{0}$ and $\mathbf{1}\in\mathcal{C}(\mathbf{X}_1)$.

(i) The coefficients of $\mathbf{X}_1$ are the same with or without $\mathbf{X}_2$.

(ii) Adding $\mathbf{X}_2$ reduces $\mathrm{SSE}$ by $\mathbf{Y}'\mathbf{X}_2(\mathbf{X}_2'\mathbf{X}_2)^{-1}\mathbf{X}_2'\mathbf{Y}$, which does not involve $\mathbf{X}_1$.

(iii) If $\mathbf{X}_2$ is one column $\mathbf{x}_2$, its variance inflation factor is $1$.

**Proof.** $\mathbf{X}'\mathbf{X}=\operatorname{diag}(\mathbf{X}_1'\mathbf{X}_1,\mathbf{X}_2'\mathbf{X}_2)$, so $\hat{\boldsymbol{\beta}}=\bigl((\mathbf{X}_1'\mathbf{X}_1)^{-1}\mathbf{X}_1'\mathbf{Y},\ (\mathbf{X}_2'\mathbf{X}_2)^{-1}\mathbf{X}_2'\mathbf{Y}\bigr)$, whose first block is the coefficient vector of the regression on $\mathbf{X}_1$ alone: (i). Likewise $\mathbf{H}=\mathbf{H}_1+\mathbf{X}_2(\mathbf{X}_2'\mathbf{X}_2)^{-1}\mathbf{X}_2'$ with $\mathbf{H}_1$ the hat matrix of $\mathbf{X}_1$, and $\mathrm{SSE}_1-\mathrm{SSE}=\mathbf{Y}'(\mathbf{H}-\mathbf{H}_1)\mathbf{Y}$: (ii). For (iii), $\mathbf{1}'\mathbf{x}_2=0$, so $x_2$ has mean $0$ and its sample covariance with any column $\mathbf{x}_l$ of $\mathbf{X}_1$ is $\mathbf{x}_2'\mathbf{x}_l/(n-1)=0$; Proposition 4.20(i) applies, and if $\mathbf{X}_1=\mathbf{1}$ the regression of $\mathbf{x}_2$ on the intercept alone has $R^2=0$, as in Unit 3's corollary **the standard error through the other explanatory variables**. ∎

**Example 4.13** (Orthogonal polynomials; FREES §5.5.5). `poly(horsepower, 3)` has orthonormal columns orthogonal to $\mathbf{1}$, as the second printed line checks, so Proposition 4.24 applies to the cubic column:

In [ ]:
p2 <- lm(mpg ~ poly(horsepower, 2), data = Auto); p3 <- lm(mpg ~ poly(horsepower, 3), data = Auto)
rbind(degree_2 = coef(p2), degree_3 = coef(p3)[1:3])
P3 <- poly(Auto$horsepower, 3)
c(max_offset_from_identity = max(abs(crossprod(P3) - diag(3))), max_col_sum = max(abs(colSums(P3))))
c(VIF = sapply(1:3, function(j) 1 / (1 - summary(lm(P3[, j] ~ P3[, -j]))$r.squared)),
  SSE_drop = sum(residuals(p2)^2) - sum(residuals(p3)^2), cubic_coef_sq = coef(p3)[[4]]^2)

The reduction in $\mathrm{SSE}$ from the cubic column, $15.59$, is the square of its coefficient, as (ii) gives for a column of unit length. FREES's principal components, a transformation $\mathbf{X}^*=\mathbf{X}\mathbf{P}$ to orthogonal columns, are of the same kind: by Unit 3's proposition **codings are reparametrisations**, the fit is unchanged and $\boldsymbol{\beta}^*=\mathbf{P}^{-1}\boldsymbol{\beta}$.

## 5 Heteroscedasticity

> **Reading.** FREES §5.7 · ISLR §3.3.3 (non-constant variance of error terms). SRM learning outcomes 2(d), 2(e).

**Definition 4.25** (Heteroscedasticity; FREES §5.7). The errors are **heteroscedastic** when A3 is replaced by $\operatorname{var}(\boldsymbol{\varepsilon})=\boldsymbol{\Omega}=\operatorname{diag}(\sigma_1^2,\ldots,\sigma_n^2)$ with the $\sigma_i^2$ not all equal: the errors remain uncorrelated and their variances vary by observation. A3 itself is **homoscedasticity**.

**Remark** (FREES §5.7). Proposition 4.9 with $\boldsymbol{\Sigma}=\boldsymbol{\Omega}$ gives the consequences: $\hat{\boldsymbol{\beta}}$ is still unbiased, its variance is FREES's (5.11),

$$
\operatorname{var}(\hat{\boldsymbol{\beta}})=(\mathbf{X}'\mathbf{X})^{-1}\Bigl(\sum_{i=1}^n\sigma_i^2\mathbf{x}_i\mathbf{x}_i'\Bigr)(\mathbf{X}'\mathbf{X})^{-1} ,
$$

while the reported $s^2(\mathbf{X}'\mathbf{X})^{-1}$ has expectation $\{\sum_i(1-h_{ii})\sigma_i^2/(n-k-1)\}(\mathbf{X}'\mathbf{X})^{-1}$, since $\operatorname{tr}(\mathbf{M}\boldsymbol{\Omega})= \sum_i(1-h_{ii})\sigma_i^2$. The Gauss–Markov theorem of Unit 2 assumes A3 and no longer applies.

### 5.1 Detecting heteroscedasticity

**Example 4.14** (A funnel; FREES §5.7.1, ISLR §3.3.3). The quadratic regression of `mpg` on `horsepower` of Example 4.3, and the same regression for $\ln\texttt{mpg}$ (ISLR Figure 3.11):

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.4)
l2 <- lm(log(mpg) ~ horsepower + I(horsepower^2), data = Auto)
op <- par(mfrow = c(1, 2), mar = c(3.2, 3.4, 1.4, 0.6), mgp = c(1.9, 0.6, 0))
for (m in list(a2, l2)) {
  plot(fitted(m), residuals(m), xlab = "fitted", ylab = "residual", pch = 16, cex = 0.4,
       main = if (identical(m, a2)) "mpg" else "log(mpg)", cex.main = 0.9)
  abline(h = 0, lty = 2)
}
par(op)

*Figure 4.7.* Residuals against fitted values for `mpg` and for $\ln\texttt{mpg}$ on `horsepower` and its square.

For `mpg` the residuals fan out as the fitted value grows, a funnel; for $\ln\texttt{mpg}$ their spread is nearly the same across the fitted range. Residuals are plotted against fitted values because the fitted value estimates the mean, with which the variance often grows.

**Definition 4.26** (The Breusch–Pagan test; FREES §5.7.1). For known vectors $\mathbf{z}_i\in\mathbb{R}^p$, the alternative is $H_a\colon\operatorname{var}(y_i)=\sigma^2+\mathbf{z}_i'\boldsymbol{\gamma}$ and the null hypothesis $H_0\colon\boldsymbol{\gamma}=\mathbf{0}$, which is homoscedasticity. FREES's procedure: (i) fit the regression and compute $e_i$; (ii) compute $e_i^{*2}=e_i^2/s^2$; (iii) regress $e_i^{*2}$ on $\mathbf{z}_i$ with an intercept; (iv) $\text{LM}=\tfrac12\,\mathrm{SSR}_z$, half the regression sum of squares of step (iii); (v) reject $H_0$ when $\text{LM}$ exceeds the $1-\alpha$ quantile of $\chi^2_p$. The same procedure with $\tilde\sigma^2=\mathrm{SSE}/n$ in place of $s^2$ in step (ii) is also used.

**Lemma 4.27** (Residuals in place of errors). For every $n$ let $\mathbf{Y}=\mathbf{X}\boldsymbol{\beta}+\boldsymbol{\varepsilon}$ with $\mathbf{X}$ of rank $k+1$ and rows $\mathbf{x}_1,\ldots,\mathbf{x}_n$, the first $n$ terms of a fixed sequence with $\|\mathbf{x}_i\|\leqslant B$, and suppose $n^{-1}\mathbf{X}'\mathbf{X}\to\mathbf{Q}_X$ positive definite. Let $\varepsilon_1,\varepsilon_2,\ldots$ be uncorrelated with mean $0$ and $\operatorname{var}(\varepsilon_i)=\sigma_i^2\leqslant C$. If $\mathbf{a}_{n1},\ldots,\mathbf{a}_{nn}$ are non-random vectors in $\mathbb{R}^q$ with $\|\mathbf{a}_{ni}\|\leqslant A$, then

$$
\frac1{\sqrt n}\sum_{i=1}^n\mathbf{a}_{ni}\,(e_i^2-\varepsilon_i^2)\xrightarrow{\,p\,}\mathbf{0} .
$$

**Proof.** Let $\boldsymbol{\delta}=\hat{\boldsymbol{\beta}}-\boldsymbol{\beta}=(\mathbf{X}'\mathbf{X})^{-1}\mathbf{X}'\boldsymbol{\varepsilon}$, so $e_i=\varepsilon_i-\mathbf{x}_i'\boldsymbol{\delta}$ and $e_i^2-\varepsilon_i^2=(\mathbf{x}_i'\boldsymbol{\delta})^2-2\varepsilon_i\mathbf{x}_i'\boldsymbol{\delta}$. The sum is $\mathbf{T}_1-2\mathbf{T}_2$ with $\mathbf{T}_1=n^{-1/2}\sum_i\mathbf{a}_{ni}(\mathbf{x}_i'\boldsymbol{\delta})^2$ and $\mathbf{T}_2=n^{-1/2}\sum_i\mathbf{a}_{ni}\varepsilon_i\mathbf{x}_i'\boldsymbol{\delta}$.

**Step 1: $\mathbf{T}_1$.** $\|\mathbf{T}_1\|\leqslant An^{-1/2}\sum_i(\mathbf{x}_i'\boldsymbol{\delta})^2 =An^{-1/2}\|\mathbf{X}\boldsymbol{\delta}\|^2=An^{-1/2}\boldsymbol{\varepsilon}'\mathbf{H}\boldsymbol{\varepsilon}$, since $\mathbf{X}\boldsymbol{\delta}=\mathbf{H}\boldsymbol{\varepsilon}$. By the prerequisites' lemma **expectation of a quadratic form**, $\operatorname{\mathsf{E}}[\boldsymbol{\varepsilon}'\mathbf{H}\boldsymbol{\varepsilon}]=\sum_ih_{ii}\sigma_i^2\leqslant C\operatorname{tr}(\mathbf{H})=C(k+1)$. So $\operatorname{\mathsf{E}}\|\mathbf{T}_1\|\leqslant AC(k+1)/\sqrt n\to0$, and $\mathbf{T}_1\xrightarrow{\,p\,}\mathbf{0}$ by Markov's inequality.

**Step 2: $\mathbf{T}_2$.** $\mathbf{T}_2=\mathbf{G}_n\boldsymbol{\delta}$ with $\mathbf{G}_n=n^{-1/2}\sum_i\varepsilon_i\mathbf{a}_{ni}\mathbf{x}_i'$, a $q\times(k+1)$ random matrix. By Cauchy–Schwarz on each row, $\|\mathbf{G}_n\boldsymbol{\delta}\|\leqslant\|\mathbf{G}_n\|_F\|\boldsymbol{\delta}\|$, where $\|\cdot\|_F$ is the root sum of squares of the entries, and $2ab\leqslant a^2+b^2$ gives $\|\mathbf{G}_n\|_F\|\boldsymbol{\delta}\|\leqslant\tfrac12(n^{-1/2}\|\mathbf{G}_n\|_F^2+n^{1/2}\|\boldsymbol{\delta}\|^2)$. The errors are uncorrelated with mean $0$, so the cross terms vanish and $\operatorname{\mathsf{E}}\|\mathbf{G}_n\|_F^2=n^{-1}\sum_i\sigma_i^2\|\mathbf{a}_{ni}\|^2\|\mathbf{x}_i\|^2\leqslant CA^2B^2$. By the quadratic-form lemma again and the cyclic property of the trace,

$$
\operatorname{\mathsf{E}}\|\boldsymbol{\delta}\|^2=\sum_i\sigma_i^2\,\mathbf{x}_i'(\mathbf{X}'\mathbf{X})^{-2}\mathbf{x}_i
\leqslant C\operatorname{tr}\bigl(\mathbf{X}(\mathbf{X}'\mathbf{X})^{-2}\mathbf{X}'\bigr)=C\operatorname{tr}\bigl((\mathbf{X}'\mathbf{X})^{-1}\bigr)
=\frac Cn\operatorname{tr}\bigl((n^{-1}\mathbf{X}'\mathbf{X})^{-1}\bigr) .
$$

By the prerequisites' lemma **inverses of convergent positive definite matrices**, the entries of $(n^{-1}\mathbf{X}'\mathbf{X})^{-1}$ are bounded for all large $n$, so $\operatorname{\mathsf{E}}\|\mathbf{T}_2\|\leqslant\tfrac12n^{-1/2}\{CA^2B^2+C\operatorname{tr}((n^{-1}\mathbf{X}'\mathbf{X})^{-1})\}\to0$. Markov's inequality gives $\mathbf{T}_2\xrightarrow{\,p\,}\mathbf{0}$. Finally $\operatorname{\mathsf{P}}\{\|\mathbf{T}_1-2\mathbf{T}_2\|>\eta\}\leqslant\operatorname{\mathsf{P}}\{\|\mathbf{T}_1\|>\eta/2\}+\operatorname{\mathsf{P}}\{2\|\mathbf{T}_2\|>\eta/2\}\to0$. ∎

**Theorem 4.28** (Null distribution of the Breusch–Pagan statistic; FREES §5.7.1). Let $\varepsilon_1,\varepsilon_2,\ldots$ be independent and identically distributed with mean $0$, variance $\sigma^2>0$, $\operatorname{\mathsf{E}}[\varepsilon_1^4]<\infty$ and $\kappa=\operatorname{var}(\varepsilon_1^2)/\sigma^4>0$. For every $n$ let $\mathbf{Y}=\mathbf{X}\boldsymbol{\beta}+\boldsymbol{\varepsilon}$, where $\mathbf{X}$ has rank $k+1$ and rows $\mathbf{x}_i$, and $\mathbf{Z}$ has rows $\mathbf{z}_i\in\mathbb{R}^p$, both the first $n$ terms of fixed sequences with $\|\mathbf{x}_i\|,\|\mathbf{z}_i\|\leqslant B$; let $\mathbf{Z}_1=[\mathbf{1}\ \mathbf{Z}]$ have rank $p+1$, and let $n^{-1}\mathbf{X}'\mathbf{X}\to\mathbf{Q}_X$ and $n^{-1}\mathbf{Z}_1'\mathbf{Z}_1\to\mathbf{Q}_1$, both positive definite. Then the statistic $\text{LM}$ of Definition 4.26, with $s^2$ or with $\tilde\sigma^2$, satisfies $(2/\kappa)\,\text{LM}\xrightarrow{\,d\,}\chi^2_p$. Under A5, $\kappa=2$ and $\text{LM}\xrightarrow{\,d\,}\chi^2_p$.

**Proof.** Write $g_i=\varepsilon_i^2/\sigma^2-1$, $\bar{\mathbf{z}}_n=n^{-1}\mathbf{Z}'\mathbf{1}$ and $\tilde{\mathbf{Z}}=\mathbf{Z}-\mathbf{1}\bar{\mathbf{z}}_n'$, whose columns sum to zero.

**Step 1: a central limit theorem.** The $g_i$ are independent and identically distributed with mean $0$ and variance $\kappa>0$, and $\sup_i\operatorname{\mathsf{E}}[g_i^2;|g_i|>a]=\operatorname{\mathsf{E}}[g_1^2;|g_1|>a]\to0$ as $a\to\infty$ by dominated convergence. The vectors $\mathbf{w}_i=(1,\mathbf{z}_i')'$ satisfy $n^{-1}\sum_i\mathbf{w}_i\mathbf{w}_i'= n^{-1}\mathbf{Z}_1'\mathbf{Z}_1\to\mathbf{Q}_1$ and $n^{-1}\max_i\|\mathbf{w}_i\|^2\leqslant(1+B^2)/n\to0$. The prerequisites' corollary **weighted sums** gives $n^{-1/2}\mathbf{Z}_1'\mathbf{g}\xrightarrow{\,d\,} N(\mathbf{0},\kappa\mathbf{Q}_1)$. Now $\tilde{\mathbf{Z}}=\mathbf{Z}_1\mathbf{L}_n'$ with $\mathbf{L}_n=[-\bar{\mathbf{z}}_n\ \ \mathbf{I}_p]$; $\bar{\mathbf{z}}_n$ is part of the first column of $n^{-1}\mathbf{Z}_1'\mathbf{Z}_1$, so it converges to some $\boldsymbol{\mu}$, and $\mathbf{L}_n\to\mathbf{L}=[-\boldsymbol{\mu}\ \ \mathbf{I}_p]$. By the prerequisites' Slutsky theorem, part (ii), with $\Phi(\mathbf{v},\mathbf{L})=\mathbf{L}\mathbf{v}$, and their theorem **closure under affine maps**,

$$
n^{-1/2}\tilde{\mathbf{Z}}'\mathbf{g}=\mathbf{L}_n\,n^{-1/2}\mathbf{Z}_1'\mathbf{g}\xrightarrow{\,d\,} N(\mathbf{0},\kappa\mathbf{Q}),\qquad
\mathbf{Q}=\mathbf{L}\mathbf{Q}_1\mathbf{L}'=\lim_n n^{-1}\tilde{\mathbf{Z}}'\tilde{\mathbf{Z}} ,
$$

and $\mathbf{Q}$ is positive definite, since $\mathbf{L}'\mathbf{v}\neq\mathbf{0}$ for $\mathbf{v}\neq\mathbf{0}$.

**Step 2: residuals for errors.** With $\hat g_i=e_i^2/\sigma^2-1$, Lemma 4.27 with $\mathbf{a}_{ni}=(\mathbf{z}_i-\bar{\mathbf{z}}_n)/\sigma^2$, of norm at most $2B/\sigma^2$, gives $n^{-1/2}\tilde{\mathbf{Z}}'(\hat{\mathbf{g}}-\mathbf{g})\xrightarrow{\,p\,}\mathbf{0}$, so $n^{-1/2}\tilde{\mathbf{Z}}'\hat{\mathbf{g}}\xrightarrow{\,d\,} N(\mathbf{0},\kappa\mathbf{Q})$ by Slutsky's theorem.

**Step 3: the variance estimate.** $\tilde\sigma^2=n^{-1}\boldsymbol{\varepsilon}'\mathbf{M}\boldsymbol{\varepsilon}= n^{-1}\boldsymbol{\varepsilon}'\boldsymbol{\varepsilon}-n^{-1}\boldsymbol{\varepsilon}'\mathbf{H}\boldsymbol{\varepsilon}$. By Chebyshev's inequality with $\operatorname{var}(n^{-1}\sum_i\varepsilon_i^2)=\kappa\sigma^4/n$, $n^{-1}\boldsymbol{\varepsilon}'\boldsymbol{\varepsilon}\xrightarrow{\,p\,}\sigma^2$; and $n^{-1}\boldsymbol{\varepsilon}'\mathbf{H}\boldsymbol{\varepsilon}\geqslant0$ has mean $\sigma^2(k+1)/n$, so it tends to $0$ in probability by Markov's inequality. Hence $\tilde\sigma^2\xrightarrow{\,p\,}\sigma^2$, and $s^2=\tilde\sigma^2\,n/(n-k-1)\xrightarrow{\,p\,}\sigma^2$. Let $\hat\sigma^2$ be either and $u_i=e_i^2/\hat\sigma^2$. Since $\tilde{\mathbf{Z}}'\mathbf{1}=\mathbf{0}$, $\tilde{\mathbf{Z}}'(\mathbf{u}-\mathbf{1})=\tilde{\mathbf{Z}}'\mathbf{u}=(\sigma^2/\hat\sigma^2)\,\tilde{\mathbf{Z}}'\hat{\mathbf{g}}$, and $\sigma^2/\hat\sigma^2\xrightarrow{\,p\,}1$ by the prerequisites' continuous mapping theorem. By Slutsky's theorem, $\mathbf{V}_n=n^{-1/2}\tilde{\mathbf{Z}}'\mathbf{u}\xrightarrow{\,d\,} N(\mathbf{0},\kappa\mathbf{Q})$.

**Step 4: the statistic.** By (2) for the design $\mathbf{Z}_1$, the regression sum of squares of step (iii) is $\|(\mathbf{H}_{\mathbf{Z}_1}-\tfrac1n\mathbf{1}\mathbf{1}')\mathbf{u}\|^2=\mathbf{u}'\tilde{\mathbf{Z}}(\tilde{\mathbf{Z}}'\tilde{\mathbf{Z}})^{-1}\tilde{\mathbf{Z}}'\mathbf{u}$, so

$$
\frac2\kappa \text{LM}=\mathbf{V}_n'\bigl(\kappa\,n^{-1}\tilde{\mathbf{Z}}'\tilde{\mathbf{Z}}\bigr)^{-1}\mathbf{V}_n .
$$

$\kappa n^{-1}\tilde{\mathbf{Z}}'\tilde{\mathbf{Z}}$ is symmetric and tends to $\kappa\mathbf{Q}$, positive definite, and the prerequisites' theorem **quadratic forms of asymptotically normal vectors** gives $(2/\kappa)\text{LM}\xrightarrow{\,d\,}\chi^2_p$. Under A5, $\varepsilon_1^2/\sigma^2\sim\chi^2_1$, whose variance is $2$ by the prerequisites' moments of $\chi^2$. ∎

**Proposition 4.29** (Versions of the statistic; FREES §5.7.1). Under A4, let $\mathbf{Z}_1=[\mathbf{1}\ \mathbf{Z}]$ have rank $p+1$ and $\mathrm{SSE}>0$, let $\text{LM}_s$ and $\text{LM}_{\tilde\sigma}$ be the statistic of Definition 4.26 with $s^2$ and with $\tilde\sigma^2$, let $\hat\tau^2=n^{-1}\sum_i(e_i^2-\tilde\sigma^2)^2>0$, and let $R_z^2$ be the coefficient of determination of the regression of $e_i^2$ on $\mathbf{z}_i$ with an intercept. Then

$$
\text{LM}_s=\Bigl(\frac{n-k-1}{n}\Bigr)^2\text{LM}_{\tilde\sigma},\qquad
\text{LM}_{\tilde\sigma}=nR_z^2\cdot\frac{\hat\tau^2}{2\tilde\sigma^4} .
$$

**Proof.** The regression sum of squares is a quadratic form in the response, so multiplying the response by $c$ multiplies it by $c^2$; and $e_i^2/s^2=\{(n-k-1)/n\}\,e_i^2/\tilde\sigma^2$. For the second identity let $v_i=e_i^2$, with regression and total sums of squares $\mathrm{SSR}_v$ and $\mathrm{SST}_v$. Then $\text{LM}_{\tilde\sigma}=\mathrm{SSR}_v/(2\tilde\sigma^4)$; $\bar v=\mathrm{SSE}/n= \tilde\sigma^2$, so $\mathrm{SST}_v=n\hat\tau^2$; and $R_z^2=\mathrm{SSR}_v/\mathrm{SST}_v$. Hence $nR_z^2\hat\tau^2/(2\tilde\sigma^4)=\mathrm{SSR}_v/(2\tilde\sigma^4)$. ∎

**Corollary 4.30** (Koenker's studentised statistic (beyond the syllabus)). Under the hypotheses of Theorem 4.28 with $\operatorname{\mathsf{E}}[\varepsilon_1^8]<\infty$, $nR_z^2\xrightarrow{\,d\,}\chi^2_p$, whatever the value of $\kappa$.

**Proof.** By Proposition 4.29, $nR_z^2=(2/\kappa)\text{LM}_{\tilde\sigma}\cdot\kappa\tilde\sigma^4/\hat\tau^2$, and the first factor tends in distribution to $\chi^2_p$ by Theorem 4.28. By Slutsky's theorem it suffices that $\hat\tau^2/\tilde\sigma^4\xrightarrow{\,p\,}\kappa$. Since $n^{-1}\sum_ie_i^2=\tilde\sigma^2$, $\hat\tau^2=n^{-1}\sum_ie_i^4-\tilde\sigma^4$, and $\tilde\sigma^4\xrightarrow{\,p\,}\sigma^4$ (Step 3 of Theorem 4.28 and continuous mapping); it remains that $n^{-1}\sum_ie_i^4\xrightarrow{\,p\,}\mu_4=\operatorname{\mathsf{E}}[\varepsilon_1^4]=(\kappa+1)\sigma^4$. First, $n^{-1}\sum_i\varepsilon_i^4\xrightarrow{\,p\,}\mu_4$ by Chebyshev's inequality, since $\operatorname{var}(\varepsilon_1^4)\leqslant\operatorname{\mathsf{E}}[\varepsilon_1^8]<\infty$. Next, with $d_i=\mathbf{x}_i'\boldsymbol{\delta}$ as in Lemma 4.27, $|d_i|\leqslant B\|\boldsymbol{\delta}\|=:\xi_n$ and $e_i^4-\varepsilon_i^4=-4\varepsilon_i^3d_i+6\varepsilon_i^2d_i^2-4\varepsilon_id_i^3+d_i^4$, so

$$
\Bigl|\frac1n\sum_i(e_i^4-\varepsilon_i^4)\Bigr|
\leqslant4\xi_nm_3+6\xi_n^2m_2+4\xi_n^3m_1+\xi_n^4,\qquad
m_r=\frac1n\sum_i|\varepsilon_i|^r .
$$

$\operatorname{\mathsf{E}}[m_r]=\operatorname{\mathsf{E}}|\varepsilon_1|^r\leqslant1+\mu_4$, and $\operatorname{\mathsf{E}}[\xi_n^2]\leqslant B^2\sigma^2\operatorname{tr}((\mathbf{X}'\mathbf{X})^{-1})=B^2\sigma^2n^{-1}\operatorname{tr}((n^{-1}\mathbf{X}'\mathbf{X})^{-1})\to0$, the entries of $(n^{-1}\mathbf{X}'\mathbf{X})^{-1}$ being bounded, so $\xi_n\xrightarrow{\,p\,}0$. A product $U_nV_n$ with $U_n\xrightarrow{\,p\,}0$ and $\operatorname{\mathsf{E}}|V_n|\leqslant K$ tends to $0$ in probability: $\operatorname{\mathsf{P}}\{|U_nV_n|>\eta\}\leqslant K/M+\operatorname{\mathsf{P}}\{|U_n|>\eta/M\}$ for every $M>0$. Each term of the bound therefore tends to $0$ in probability, and $\hat\tau^2/\tilde\sigma^4\xrightarrow{\,p\,}(\mu_4-\sigma^4)/\sigma^4=\kappa$. ∎

**Example 4.15** (Testing the funnel; FREES §5.7.1). With $\mathbf{z}_i$ the two explanatory variables, FREES's five steps with $s^2$ and with $\tilde\sigma^2$, against `bptest`, for `mpg` and for $\ln\texttt{mpg}$:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
library(lmtest)
bp_versions <- function(m) {
  e <- residuals(m); nn <- length(e); kk <- length(coef(m)) - 1
  Z <- model.matrix(m)[, -1, drop = FALSE]; s2t <- sum(e^2) / nn; s2 <- sum(e^2) / (nn - kk - 1)
  ssr <- function(u) sum((fitted(lm(u ~ Z)) - mean(u))^2)
  c(LM_s = ssr(e^2 / s2) / 2, LM_tilde = ssr(e^2 / s2t) / 2,
    bptest_studentize_FALSE = unname(bptest(m, studentize = FALSE)$statistic),
    nR2 = nn * summary(lm(e^2 ~ Z))$r.squared, bptest_default = unname(bptest(m)$statistic),
    tau2_over_2sigma4 = mean((e^2 - s2t)^2) / (2 * s2t^2), factor_s = ((nn - kk - 1) / nn)^2,
    p_LM_tilde = pchisq(ssr(e^2 / s2t) / 2, ncol(Z), lower.tail = FALSE),
    p_nR2 = pchisq(nn * summary(lm(e^2 ~ Z))$r.squared, ncol(Z), lower.tail = FALSE))
}
round(rbind(mpg = bp_versions(a2), log_mpg = bp_versions(l2)), 4)

For `mpg`, $\text{LM}_{\tilde\sigma}=56.95$, which `bptest(\ldots, studentize = FALSE)` reproduces, and $\text{LM}_s=56.09$ is smaller by the factor $0.985$ of Proposition 4.29; either rejects homoscedasticity at any usual level against $\chi^2_2$. `bptest`'s default is $nR_z^2=34.53$, the ratio $\hat\tau^2/(2\tilde\sigma^4)=1.65$ being the measured departure of the fourth moment of the residuals from the normal value. For $\ln\texttt{mpg}$ the statistic falls to $7.47$ ($p=0.024$) and $nR_z^2$ to $5.20$ ($p=0.074$).

**Example 4.16** (The null distribution; FREES §5.7.1). Theorem 4.28 says that $\text{LM}$ tends to $(\kappa/2)\chi^2_p$, so its test has asymptotic level $\operatorname{\mathsf{P}}\{\chi^2_p>(2/\kappa)\chi^2_{p,0.95}\}$, which is $0.05$ only when $\kappa=2$. Simulated rejection rates at the nominal $5\%$ with $n=200$, $p=1$, and errors that are normal ($\kappa=2$), centred exponential ($\kappa=8$) and uniform ($\kappa=0.8$):

In [ ]:
set.seed(2); nb <- 200; xb <- runif(nb); Xb <- cbind(1, xb)
Mb <- diag(nb) - Xb %*% solve(crossprod(Xb)) %*% t(Xb); zc <- xb - mean(xb)
gen <- list(normal = function() rnorm(nb), exponential = function() rexp(nb) - 1,
            uniform = function() runif(nb, -sqrt(3), sqrt(3)))
kap <- c(normal = 2, exponential = 8, uniform = 0.8)
round(t(sapply(names(gen), function(g) {
  st <- replicate(2000, {
    e <- drop(Mb %*% gen[[g]]()); u <- e^2 / mean(e^2)
    c(LM = sum(zc * u)^2 / sum(zc^2) / 2,
      nR2 = nb * sum(zc * e^2)^2 / sum(zc^2) / sum((e^2 - mean(e^2))^2))
  })
  c(kappa = kap[[g]], LM_reject = mean(st[1, ] > qchisq(0.95, 1)),
    limit = pchisq(qchisq(0.95, 1) * 2 / kap[[g]], 1, lower.tail = FALSE),
    nR2_reject = mean(st[2, ] > qchisq(0.95, 1)))
})), 3)

The statistic $\text{LM}$ rejects a true null $4.6\%$ of the time with normal errors, $29\%$ with exponential errors (limit $33\%$) and $0.4\%$ with uniform errors (limit $0.2\%$); Koenker's $nR_z^2$ stays near $5\%$ in all three.

### 5.2 Heteroscedasticity-consistent standard errors

**Definition 4.31** (Robust standard errors; FREES §5.7.2). With weights $c_i>0$, let

$$
\hat{\mathbf{V}}_c=(\mathbf{X}'\mathbf{X})^{-1}\Bigl(\sum_{i=1}^nc_i\,e_i^2\mathbf{x}_i\mathbf{x}_i'\Bigr)(\mathbf{X}'\mathbf{X})^{-1},
\qquad \mathrm{SE}_r(\hat\beta_j)=\sqrt{[\hat{\mathbf{V}}_c]_{jj}} .
$$

FREES's **empirical** or **robust** estimator is HC0, $c_i=1$; HC1 takes $c_i=n/(n-k-1)$, HC2 $c_i=1/(1-h_{ii})$ and HC3 $c_i=1/(1-h_{ii})^2$, the last two built from the standardised and the deleted residuals ($h_{ii}<1$).

**Proposition 4.32** (Expected squared residuals; FREES §5.7.2). Under A1, A2 and A4 with uncorrelated errors, $\operatorname{var}(\varepsilon_i)= \sigma_i^2$,

$$
\operatorname{\mathsf{E}}[e_i^2]=(1-2h_{ii})\sigma_i^2+\sum_{j=1}^nh_{ij}^2\sigma_j^2 .
$$

Under homoscedasticity $\operatorname{\mathsf{E}}[e_i^2]=(1-h_{ii})\sigma^2$, so $\operatorname{\mathsf{E}}[e_i^2/(1-h_{ii})]= \sigma^2$: the middle matrix of HC2 is then unbiased for $\sum_i\sigma_i^2\mathbf{x}_i\mathbf{x}_i'$, that of HC0 is biased downward, and that of HC3 upward.

**Proof.** $\mathbf e=\mathbf{M}\boldsymbol{\varepsilon}$, so $e_i=\sum_jm_{ij}\varepsilon_j$ with $m_{ii}=1-h_{ii}$ and $m_{ij}=-h_{ij}$, and with mean-zero uncorrelated errors $\operatorname{\mathsf{E}}[e_i^2]=\sum_jm_{ij}^2\sigma_j^2=(1-h_{ii})^2\sigma_i^2+\sum_{j\neq i}h_{ij}^2\sigma_j^2$. Expanding $(1-h_{ii})^2=1-2h_{ii}+h_{ii}^2$ and moving $h_{ii}^2\sigma_i^2$ into the sum gives the formula. With $\sigma_j^2=\sigma^2$, $\sum_jh_{ij}^2=h_{ii}$ (from $\mathbf{H}=\mathbf{H}^2$, Unit 2's theorem **properties of leverage**) gives $\sigma^2(1-h_{ii})$. Then $\operatorname{\mathsf{E}}[e_i^2]=(1-h_{ii})\sigma^2\leqslant\sigma^2\leqslant\sigma^2/(1-h_{ii})= \operatorname{\mathsf{E}}[e_i^2/(1-h_{ii})^2]$, and each middle matrix is a sum of the corresponding multiples of $\mathbf{x}_i\mathbf{x}_i'$. ∎

**Theorem 4.33** (Consistency of the robust standard errors; FREES §5.7.2). For every $n$ let $\mathbf{Y}=\mathbf{X}\boldsymbol{\beta}+\boldsymbol{\varepsilon}$ with $\mathbf{X}$ of rank $k+1$ and rows $\mathbf{x}_i$ from a fixed sequence, $\|\mathbf{x}_i\|\leqslant B$; let the errors be independent with mean $0$, $\operatorname{var}(\varepsilon_i)=\sigma_i^2$ and $\sup_i\operatorname{\mathsf{E}}[\varepsilon_i^4]\leqslant C$; and let $n^{-1}\mathbf{X}'\mathbf{X}\to\mathbf{Q}_X$ and $n^{-1}\sum_i\sigma_i^2\mathbf{x}_i\mathbf{x}_i'\to\mathbf{F}$, both positive definite. Then

(i) $n^{-1}\sum_ie_i^2\mathbf{x}_i\mathbf{x}_i'-n^{-1}\sum_i\sigma_i^2\mathbf{x}_i\mathbf{x}_i'\xrightarrow{\,p\,}\mathbf{0}$;

(ii) for every $\mathbf{c}\neq\mathbf{0}$, $(\mathbf{c}'\hat{\boldsymbol{\beta}}-\mathbf{c}'\boldsymbol{\beta})/\sqrt{\mathbf{c}'\hat{\mathbf{V}}_c\mathbf{c}}\xrightarrow{\,d\,} N(0,1)$ for HC0;

(iii) (ii) holds for HC1, HC2 and HC3.

**Proof.** **(i)** Split the difference into $n^{-1}\sum_i(e_i^2-\varepsilon_i^2)\mathbf{x}_i\mathbf{x}_i'$ and $n^{-1}\sum_i(\varepsilon_i^2-\sigma_i^2)\mathbf{x}_i\mathbf{x}_i'$. Since $\sigma_i^2\leqslant1+\operatorname{\mathsf{E}}[\varepsilon_i^4] \leqslant1+C$, Lemma 4.27 with $\mathbf{a}_{ni}$ the entries of $\mathbf{x}_i\mathbf{x}_i'$, of norm at most $B^2$, shows that $n^{1/2}$ times the first tends to $\mathbf{0}$ in probability, hence so does the first. Each entry of the second has mean $0$ and variance at most $n^{-2}\sum_i\operatorname{\mathsf{E}}[\varepsilon_i^4]B^4\leqslant CB^4/n$, so it tends to $0$ in probability by Chebyshev's inequality.

**(ii)** $\sqrt n(\hat{\boldsymbol{\beta}}-\boldsymbol{\beta})=(n^{-1}\mathbf{X}'\mathbf{X})^{-1}n^{-1/2}\sum_i\mathbf{x}_i\varepsilon_i$. The vectors $\mathbf{x}_i\varepsilon_i$ are independent with mean $\mathbf{0}$, $\operatorname{\mathsf{E}}\|\mathbf{x}_i\varepsilon_i\|^4\leqslant B^4C$ and $n^{-1}\sum_i\operatorname{var}(\mathbf{x}_i\varepsilon_i)=n^{-1}\sum_i\sigma_i^2\mathbf{x}_i\mathbf{x}_i'\to\mathbf{F}$, so the prerequisites' Lyapunov corollary with $\delta=2$ gives $n^{-1/2}\sum_i\mathbf{x}_i\varepsilon_i\xrightarrow{\,d\,} N(\mathbf{0},\mathbf{F})$. By the prerequisites' lemma **inverses of convergent positive definite matrices**, $(n^{-1}\mathbf{X}'\mathbf{X})^{-1}\to\mathbf{Q}_X^{-1}$, and Slutsky's theorem with the affine-map theorem gives $\sqrt n\,\mathbf{c}'(\hat{\boldsymbol{\beta}}-\boldsymbol{\beta})\xrightarrow{\,d\,} N(0,v)$, $v=\mathbf{c}'\mathbf{Q}_X^{-1}\mathbf{F}\mathbf{Q}_X^{-1}\mathbf{c}>0$. By (i) and continuous mapping, $n\,\mathbf{c}'\hat{\mathbf{V}}_c\mathbf{c}=\mathbf{c}'(n^{-1}\mathbf{X}'\mathbf{X})^{-1}(n^{-1}\sum_ie_i^2\mathbf{x}_i\mathbf{x}_i') (n^{-1}\mathbf{X}'\mathbf{X})^{-1}\mathbf{c}\xrightarrow{\,p\,} v$, and the ratio tends to $N(0,1)$ by Slutsky's theorem.

**(iii)** For any symmetric $\mathbf{A}$ and vector $\mathbf{x}$, $|\mathbf{x}'\mathbf{A}\mathbf{x}|\leqslant\|\mathbf{x}\|^2\sum_{r,s}|a_{rs}|$; hence $\max_ih_{ii}\leqslant\eta_n=(B^2/n)\sum_{r,s}|[(n^{-1}\mathbf{X}'\mathbf{X})^{-1}]_{rs}|\to0$, the entries being bounded by the same lemma. The weights of HC1 to HC3 lie between $1$ and $\max\{n/(n-k-1),(1-\eta_n)^{-2}\}\to1$, so the difference of their middle matrices from that of HC0, divided by $n$, has entries at most $(\max_ic_i-1)\,B^2\,\mathrm{SSE}/n$, where $\operatorname{\mathsf{E}}[\mathrm{SSE}/n]\leqslant n^{-1}\sum_i\sigma_i^2\leqslant1+C$ (Proposition 4.9); it tends to $\mathbf{0}$ in probability, and the argument of (ii) applies. ∎

**Example 4.17** (Robust standard errors; FREES §5.7.2). The four estimators from Definition 4.31 for the quadratic `mpg` fit, against `sandwich::vcovHC`, and the coverage of nominal $95\%$ intervals for a slope when $\operatorname{sd}(\varepsilon_i)=x_i^2$ with $x_i$ uniform on $(0,3)$:

In [ ]:
hc_manual <- function(m, type) {
  X <- model.matrix(m); e <- residuals(m); h <- hatvalues(m); XtXi <- solve(crossprod(X))
  cw <- switch(type, HC0 = 1, HC1 = nrow(X) / (nrow(X) - ncol(X)), HC2 = 1 / (1 - h), HC3 = 1 / (1 - h)^2)
  sqrt(diag(XtXi %*% crossprod(X * sqrt(cw) * e) %*% XtXi))
}
types <- c("HC0", "HC1", "HC2", "HC3")
se_tab <- cbind(usual = sqrt(diag(vcov(a2))), sapply(types, function(tp) hc_manual(a2, tp)))
round(se_tab, 6)
round(se_tab[, types] / se_tab[, "usual"], 3)
c(max_diff_sandwich = max(sapply(types, function(tp)
  max(abs(hc_manual(a2, tp) - sqrt(diag(sandwich::vcovHC(a2, type = tp))))))))
set.seed(3)
cover <- function(nn, R = 4000) {
  x <- runif(nn, 0, 3); X <- cbind(1, x); XtXi <- solve(crossprod(X)); h <- rowSums((X %*% XtXi) * X)
  res <- replicate(R, {
    y <- 1 + 2 * x + x^2 * rnorm(nn); b <- XtXi %*% crossprod(X, y); e <- drop(y - X %*% b)
    V0 <- XtXi %*% crossprod(X * e) %*% XtXi; V3 <- XtXi %*% crossprod(X * (e / (1 - h))) %*% XtXi
    q <- qt(0.975, nn - 2)
    c(usual = abs(b[2] - 2) <= q * sqrt(sum(e^2) / (nn - 2) * XtXi[2, 2]),
      HC0 = abs(b[2] - 2) <= q * sqrt(V0[2, 2]), HC3 = abs(b[2] - 2) <= q * sqrt(V3[2, 2]))
  })
  c(n = nn, max_h = max(h), rowMeans(res))
}
round(t(sapply(c(20, 50, 200, 1000), cover)), 3)

For the quadratic fit the robust standard errors exceed the usual ones by $15\%$ to $17\%$ for the intercept and $6\%$ to $8\%$ for the linear term. In the simulation the usual interval covers the slope $88\%$ to $90\%$ of the time at every $n$, since it estimates the wrong variance; the HC0 interval rises from $89\%$ to the nominal level as $\max_ih_{ii}$ falls, as Theorem 4.33 predicts, and HC3 is close to $95\%$ already at $n=20$, in line with FREES's advice to prefer an estimator built from rescaled residuals.

### 5.3 Weighted least squares

**Theorem 4.34** (Weighted least squares; FREES §5.7.3, ISLR §3.3.3). Assume A1, A2, A4 and $\operatorname{var}(\boldsymbol{\varepsilon})=\sigma^2\mathbf{W}^{-1}$ with $\sigma^2>0$, $\mathbf{W}=\operatorname{diag}(w_1,\ldots,w_n)$ and known $w_i>0$. Let $\mathbf{Y}^*=\mathbf{W}^{1/2}\mathbf{Y}$ and $\mathbf{X}^*=\mathbf{W}^{1/2}\mathbf{X}$, the rows multiplied by $\sqrt{w_i}$.

(i) Least squares on $(\mathbf{X}^*,\mathbf{Y}^*)$ gives $\hat{\boldsymbol{\beta}}_W=(\mathbf{X}'\mathbf{W}\mathbf{X})^{-1}\mathbf{X}'\mathbf{W}\mathbf{Y}$, the minimiser of $\sum_iw_i(y_i-\mathbf{x}_i'\boldsymbol{\beta})^2$.

(ii) $\hat{\boldsymbol{\beta}}_W$ is unbiased with $\operatorname{var}(\hat{\boldsymbol{\beta}}_W)=\sigma^2(\mathbf{X}'\mathbf{W}\mathbf{X})^{-1}$ and is the best linear unbiased estimator of $\boldsymbol{\beta}$.

(iii) $s_W^2=\sum_iw_i(y_i-\mathbf{x}_i'\hat{\boldsymbol{\beta}}_W)^2/(n-k-1)$ is unbiased for $\sigma^2$, and with normal errors the $t$ procedures of Unit 2 and the general linear hypothesis $F$ test of Unit 3 applied to $(\mathbf{X}^*,\mathbf{Y}^*)$ are exact; the test of $\beta_1=\cdots=\beta_k=0$ is the latter with $\mathbf{R}=[\mathbf{0}\ \mathbf{I}_k]$, since $\mathbf{X}^*$ has no constant column.

**Proof.** $\mathbf{Y}^*=\mathbf{X}^*\boldsymbol{\beta}+\boldsymbol{\varepsilon}^*$ with $\boldsymbol{\varepsilon}^*=\mathbf{W}^{1/2}\boldsymbol{\varepsilon}$, $\operatorname{\mathsf{E}}[\boldsymbol{\varepsilon}^*]=\mathbf{0}$ and, by the prerequisites' proposition **variance under a linear map**, $\operatorname{var}(\boldsymbol{\varepsilon}^*)=\mathbf{W}^{1/2}\sigma^2\mathbf{W}^{-1}\mathbf{W}^{1/2}=\sigma^2\mathbf{I}_n$; $\mathbf{X}^*$ has full column rank since $\mathbf{W}^{1/2}$ is invertible. So $(\mathbf{X}^*,\mathbf{Y}^*)$ satisfies A1–A4 except that its first column is $(\sqrt{w_1},\ldots,\sqrt{w_n})'$ rather than $\mathbf{1}$; the theorems of Units 2 and 3 cited in this proof use only $\operatorname{rank}(\mathbf{X}^*)=k+1$ and the moments, or the normality, of $\boldsymbol{\varepsilon}^*$, not a constant column. (i) Its least squares estimator is $(\mathbf{X}^{*\prime}\mathbf{X}^*)^{-1}\mathbf{X}^{*\prime}\mathbf{Y}^*= (\mathbf{X}'\mathbf{W}\mathbf{X})^{-1}\mathbf{X}'\mathbf{W}\mathbf{Y}$, and $\|\mathbf{Y}^*-\mathbf{X}^*\boldsymbol{\beta}\|^2=\sum_iw_i(y_i-\mathbf{x}_i'\boldsymbol{\beta})^2$. (ii) Unit 2's theorems **unbiasedness**, **variance–covariance matrix** and **Gauss–Markov** for the starred model; an estimator $\mathbf{C}\mathbf{Y}$ linear in $\mathbf{Y}$ is $\mathbf{C}\mathbf{W}^{-1/2}\mathbf{Y}^*$, linear in $\mathbf{Y}^*$, so the comparison class is the same. (iii) The residuals of the starred fit are $\mathbf{W}^{1/2}(\mathbf{Y}-\mathbf{X}\hat{\boldsymbol{\beta}}_W)$, whose squared length is $\sum_iw_i(y_i-\mathbf{x}_i'\hat{\boldsymbol{\beta}}_W)^2$; Unit 2's theorem **unbiasedness of the mean square error** applies. With $\boldsymbol{\varepsilon}\sim N(\mathbf{0},\sigma^2\mathbf{W}^{-1})$, $\boldsymbol{\varepsilon}^*\sim N(\mathbf{0},\sigma^2\mathbf{I}_n)$ by the prerequisites' theorem **closure under affine maps**, and A5 holds for the starred model, so Unit 2's theorem **the $t$ statistic** and Unit 3's theorem **the general linear hypothesis** apply to $(\mathbf{X}^*,\mathbf{Y}^*)$. ∎

**Corollary 4.35** (Averages of raw observations; ISLR §3.3.3). Let observation $g=1,\ldots,G$ be the average $\bar y_g$ of $n_g$ raw responses $y_{g1},\ldots,y_{gn_g}$ that share the row $\mathbf{x}_g'$ and satisfy $y_{gl}=\mathbf{x}_g'\boldsymbol{\beta}+\varepsilon_{gl}$ with uncorrelated errors of mean $0$ and variance $\sigma^2>0$, and let the $G\times(k+1)$ matrix of the $\mathbf{x}_g'$ have rank $k+1$. Then $\operatorname{var}(\bar y_g)=\sigma^2/n_g$, so $w_g=n_g$; the weighted least squares coefficients on the $G$ averages equal the least squares coefficients on the $N=\sum_gn_g$ raw observations; and the raw residual sum of squares is $\sum_g\sum_l(y_{gl}-\bar y_g)^2+\sum_gn_g(\bar y_g-\mathbf{x}_g'\hat{\boldsymbol{\beta}})^2$.

**Proof.** $\operatorname{var}(\bar y_g)=n_g^{-2}\sum_l\sigma^2=\sigma^2/n_g$, and averages of disjoint sets of uncorrelated errors are uncorrelated, so Theorem 4.34 applies with $w_g=n_g$. The raw design repeats $\mathbf{x}_g'$ $n_g$ times, so $\mathbf{X}_{\mathrm{raw}}'\mathbf{X}_{\mathrm{raw}}=\sum_gn_g\mathbf{x}_g\mathbf{x}_g'=\mathbf{X}'\mathbf{W}\mathbf{X}$ and $\mathbf{X}_{\mathrm{raw}}'\mathbf{Y}_{\mathrm{raw}}=\sum_g\mathbf{x}_g\sum_ly_{gl}=\sum_gn_g\mathbf{x}_g\bar y_g= \mathbf{X}'\mathbf{W}\bar{\mathbf{Y}}$: the normal equations coincide. Finally $y_{gl}-\mathbf{x}_g'\hat{\boldsymbol{\beta}}=(y_{gl}-\bar y_g)+(\bar y_g-\mathbf{x}_g'\hat{\boldsymbol{\beta}})$, and the cross term vanishes on summing over $l$ because $\sum_l(y_{gl}-\bar y_g)=0$. ∎

**Example 4.18** (Fuel efficiency by model year; ISLR §3.3.3, FREES §5.7.3). The $392$ cars of `Auto` fall into $13$ model years. Weighted least squares on the yearly means with weights the counts, three ways, against least squares on the raw data:

In [ ]:
ym <- aggregate(mpg ~ year, data = Auto, FUN = mean); ym$n <- as.vector(table(Auto$year)); nrow(ym)
w_lm <- lm(mpg ~ year, data = ym, weights = n)
Xg <- cbind(1, ym$year); Wg <- diag(ym$n)
w_formula <- drop(solve(t(Xg) %*% Wg %*% Xg, t(Xg) %*% Wg %*% ym$mpg))
w_scaled <- coef(lm(I(sqrt(n) * mpg) ~ 0 + I(sqrt(n)) + I(sqrt(n) * year), data = ym))
raw <- lm(mpg ~ year, data = Auto)
rbind(lm_weights = coef(w_lm), formula = w_formula, sqrt_w_scaled = unname(w_scaled),
      raw_data = coef(raw), unweighted_means = coef(lm(mpg ~ year, data = ym)))
range(ym$n)
SS_within <- sum((Auto$mpg - ave(Auto$mpg, Auto$year))^2)
c(SSE_raw = sum(residuals(raw)^2), within_plus_weighted = SS_within + sum(ym$n * residuals(w_lm)^2))

The three weighted computations and the raw regression agree, while unweighted least squares on the means, which treats a year of $26$ cars like a year of $40$, gives a different line.

### 5.4 Transformations

**Proposition 4.36** (Concave transformations and a spread that grows with the mean; FREES §5.7.4, ISLR §3.3.3). Let $h$ be increasing and convex on an interval $I$, let $\varepsilon$ be a random variable, and for each $\mu$ in an interval $J$ with $\mu+\varepsilon\in I$ almost surely let $Y_\mu=h(\mu+\varepsilon)$ with $\operatorname{\mathsf{E}}[Y_\mu^2]<\infty$. Then $\operatorname{\mathsf{E}}[Y_\mu]$ and $\operatorname{var}(Y_\mu)$ are nondecreasing in $\mu\in J$.

**Proof.** $\operatorname{\mathsf{E}}[Y_\mu]$ is nondecreasing because $h$ is increasing. Let $\varepsilon'$ be an independent copy of $\varepsilon$; then $\operatorname{var}(Y_\mu)=\tfrac12\operatorname{\mathsf{E}}[\{h(\mu+\varepsilon)-h(\mu+\varepsilon')\}^2]$, by expanding the square. Fix $a<b$ and $\mu<\mu'$ in $J$ with the four points below in $I$, and put $x_1=\mu+a$, $x_2=\mu'+b$ and $\{y_1,y_2\}=\{\mu+b,\mu'+a\}$. Then $x_1\leqslant y_j\leqslant x_2$ and $y_1+y_2=x_1+x_2$, so $y_j=\lambda_jx_1+(1-\lambda_j)x_2$ with $\lambda_1+\lambda_2=1$, and adding the two convexity inequalities $h(y_j)\leqslant\lambda_jh(x_1)+(1-\lambda_j)h(x_2)$ gives $h(y_1)+h(y_2)\leqslant h(x_1)+h(x_2)$, that is

$$
0\leqslant h(\mu+b)-h(\mu+a)\leqslant h(\mu'+b)-h(\mu'+a) ,
$$

the left inequality because $h$ is increasing. Hence $|h(\mu+\varepsilon)-h(\mu+\varepsilon')|$ is nondecreasing in $\mu$ for every outcome, and taking expectations of its square completes the proof. ∎

**Remark** (FREES §5.7.4, ISLR §3.3.3). If $g=h^{-1}$ is increasing and concave, as $\ln y$ is on $(0,\infty)$ and $\ln(1+y)$ and $\sqrt y$ are on $[0,\infty)$, and $g(y_i)=\mathbf{x}_i'\boldsymbol{\beta}+\varepsilon_i$ with identically distributed errors, then the variance of $y_i$ does not decrease as its mean increases: that is the funnel that such a transformation of the response removes. An increasing convex $g$ can only remove a spread that does not increase with the mean: $h=g^{-1}$ is then increasing and concave, and Proposition 4.36 applied to the increasing convex $t\mapsto-h(-t)$ and to $-\varepsilon$ makes $\operatorname{var}(y_i)=\operatorname{var}(-y_i)$ nonincreasing in the mean. A transformation also acts on the whole shape of the distribution, compressing the right tail when it is concave. $\ln(1+y)$ admits zeros, and FREES's $100\ln(1+y/100)$ keeps percentage data near their original units.

**Remark** (Limits of transformation; FREES §5.7.4). When $g(y_i)$ has identically distributed errors, a concave or convex $g$ (replace $g$ by $-g$ if it is decreasing) cannot remove a spread that rises and then falls with the mean, since by Proposition 4.36 and the remark after it each can equalise only a spread that moves one way with the mean; such a pattern needs a transformation that is neither concave nor convex, such as the logit for proportions (concave below $\tfrac12$, convex above). When the data are roughly symmetric but heteroscedastic, a transformation that equalises the spread is not linear, and so in general skews the distribution. Explanatory variables may be transformed too, since the regression assumptions condition on them: some analysts transform them towards normality, others avoid it because the fitted model is harder to interpret. FREES takes a middle course: it uses transformations that are readily interpreted, such as the logarithms, powers and indicators of its §3.5, and admits a transform that is hard to communicate only when it brings a substantial gain in fit or prediction.

**Proposition 4.37** (Multiplicative errors; FREES §5.7.4). Let $y_i=\exp(\mathbf{x}_i'\boldsymbol{\beta})\,\eta_i$, with an intercept in $\mathbf{x}_i$ and $\eta_1,\ldots,\eta_n$ positive, not almost surely constant, independent and identically distributed, $\operatorname{\mathsf{E}}[\eta_1^2]<\infty$ and $\operatorname{\mathsf{E}}[(\ln\eta_1)^2]<\infty$. Then $\operatorname{sd}(y_i)/\operatorname{\mathsf{E}}[y_i]=\operatorname{sd}(\eta_1)/\operatorname{\mathsf{E}}[\eta_1]$ for every $i$, while $\ln y_i=\mathbf{x}_i'\boldsymbol{\beta}+\ln\eta_i$ satisfies A1–A3 with intercept $\beta_0+\operatorname{\mathsf{E}}[\ln\eta_1]$ and errors $\ln\eta_i-\operatorname{\mathsf{E}}[\ln\eta_1]$.

**Proof.** $\operatorname{\mathsf{E}}[y_i]=e^{\mathbf{x}_i'\boldsymbol{\beta}}\operatorname{\mathsf{E}}[\eta_1]$ and $\operatorname{sd}(y_i)=e^{\mathbf{x}_i'\boldsymbol{\beta}}\operatorname{sd}(\eta_1)$. The logarithm is $\mathbf{x}_i'\boldsymbol{\beta}+\operatorname{\mathsf{E}}[\ln\eta_1]+(\ln\eta_i-\operatorname{\mathsf{E}}[\ln\eta_1])$, with independent errors of mean $0$ and common variance $\operatorname{var}(\ln\eta_1)$. ∎

**Remark** (FREES §5.7.4). The standard deviation proportional to the mean is the funnel of Example 4.14, and the logarithm removes it exactly under Proposition 4.37; the Breusch–Pagan statistic of Example 4.15 fell from $56.95$ to $7.47$.

## 6 Computation in R

> **Reading.** ISLR §3.6.2–3.6.3 · FREES chapter 5 script.

### 6.1 The ISLR lab

`plot` applied to an `lm` fit draws four diagnostic plots:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3.3)
lm.fit <- lm(medv ~ lstat, data = Boston)
op <- par(mfrow = c(2, 2), mar = c(2.6, 3.4, 1.8, 0.6), mgp = c(1.9, 0.6, 0), cex.main = 0.8)
plot(lm.fit, cex = 0.4)
par(op)

*Figure 4.8.* The four diagnostic plots of `medv` on `lstat`.

Residuals against fitted values show curvature, the check of Example 4.3; the normal quantile plot of the $r_i$ checks A5; the scale–location plot, $\sqrt{|r_i|}$ against the fitted values, checks homoscedasticity; and the plot of $r_i$ against $h_{ii}$ draws, where they fall inside it, the contours on which Cook's distance equals $0.5$ and $1$, which by Theorem 4.14 are the curves $r=\pm\sqrt{(k+1)D(1-h)/h}$; here the largest Cook's distance is below $0.1$ and no contour appears. The lab's residual plot of $t_i$, its largest leverage, and the Bonferroni test of Corollary 4.6 as `car::outlierTest` computes it:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.4)
op <- par(mar = c(3.2, 3.4, 0.6, 0.6), mgp = c(1.9, 0.6, 0))
plot(predict(lm.fit), rstudent(lm.fit), xlab = "fitted", ylab = "studentised residual", pch = 16, cex = 0.4)
par(op)
c(which_max_h = which.max(hatvalues(lm.fit)), max_h = max(hatvalues(lm.fit)),
  max_D = max(cooks.distance(lm.fit)), n_abs_t_above_3 = sum(abs(rstudent(lm.fit)) > 3))
t_max <- max(abs(rstudent(lm.fit))); n_B <- nobs(lm.fit)
c(t_max = t_max, bonferroni_p = min(1, n_B * 2 * pt(-t_max, n_B - 1 - 2)))
car::outlierTest(lm.fit)

Eleven observations have $|t_i|>3$, ISLR's flag, and the largest, $t=4.00$ at observation $372$, has Bonferroni $p$-value $0.036$. The variance inflation factors of the full regression:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
lm.fit.all <- lm(medv ~ ., data = Boston)
round(car::vif(lm.fit.all), 2)

All are below $10$; the largest are `tax` ($9.00$) and `rad` ($7.45$).

### 6.2 FREES's chapter 5 script

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3.3)
op <- par(mfrow = c(2, 2), mar = c(2.6, 3.4, 1.8, 0.6), mgp = c(1.9, 0.6, 0), cex.main = 0.8)
plot(mv, cex = 0.4)
par(op)
c(hat_60 = hatvalues(mv)[[60]], residual_60 = residuals(mv)[[60]])

*Figure 4.9.* The four diagnostic plots of `VOLUME` on `PRICE`, `SHARE` and `VALUE`.

In the fourth panel IBM, observation $60$, lies beyond the contour $D=1$ at leverage $0.88$ with a moderate residual; observation $122$ is beyond $D=1$ as well.

### 6.3 Formulas against package output

For the liquidity regression `mv`: the leverages from $\mathbf{H}$ and from the Mahalanobis form (3); $r_i$ and $t_i$ from Definition 4.4, from (1) and from the indicator regression of Proposition 4.3; the deleted coefficients from Proposition 4.3(iii) against refitting and `dfbeta`; Cook's distance from its definition and from Theorem 4.14; and Proposition 4.16.

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
X <- model.matrix(mv); e <- residuals(mv); nX <- nrow(X); kX <- ncol(X) - 1
XtXi <- solve(crossprod(X)); H <- X %*% XtXi %*% t(X); h <- diag(H); s <- summary(mv)$sigma
Delta2 <- mahalanobis(X[, -1], colMeans(X[, -1]), cov(X[, -1]))
r <- e / (s * sqrt(1 - h))
s_del <- sqrt(((nX - kX - 1) * s^2 - e^2 / (1 - h)) / (nX - kX - 2))
t_i <- e / (s_del * sqrt(1 - h))
t_indicator <- sapply(c(37, 60, 122), function(i) {
  u <- as.numeric(seq_len(nX) == i); coef(summary(lm(VOLUME ~ PRICE + SHARE + VALUE + u, data = liq)))["u", 3]
})
signif(c(h_vs_hatvalues = max(abs(h - hatvalues(mv))), h_vs_mahalanobis = max(abs(h - (1 / nX + Delta2 / (nX - 1)))),
  r_vs_rstandard = max(abs(r - rstandard(mv))), t_vs_rstudent = max(abs(t_i - rstudent(mv))),
  t_from_r = max(abs(r * sqrt((nX - kX - 2) / (nX - kX - 1 - r^2)) - rstudent(mv))),
  t_indicator = max(abs(t_indicator - rstudent(mv)[c(37, 60, 122)]))), 3)
db_formula <- t(XtXi %*% t(X) %*% diag(e / (1 - h)))
db_refit <- t(sapply(seq_len(nX), function(i) coef(mv) - coef(lm(VOLUME ~ PRICE + SHARE + VALUE, data = liq[-i, ]))))
D_def <- sapply(seq_len(nX), function(i) sum((X %*% (coef(mv) - db_refit[i, ]) - fitted(mv))^2)) / ((kX + 1) * s^2)
signif(c(dfbeta_formula_vs_refit = max(abs(db_formula - db_refit)),
  dfbeta_vs_R = max(abs(db_formula - dfbeta(mv))),
  cook_definition_vs_R = max(abs(D_def - cooks.distance(mv))),
  cook_formula_vs_R = max(abs(r^2 / (kX + 1) * h / (1 - h) - cooks.distance(mv))),
  dffits_vs_R = max(abs(t_i * sqrt(h / (1 - h)) - dffits(mv))),
  dfbetas_vs_R = max(abs(db_formula / outer(s_del, sqrt(diag(XtXi))) - dfbetas(mv))),
  cook_from_dffits = max(abs(s_del^2 / ((kX + 1) * s^2) * dffits(mv)^2 - cooks.distance(mv)))), 3)

Every difference is of the order of rounding error.

## 7 Exercises

**Exercise 4.1** (FREES exercise 5.1). In the regression on one explanatory variable, $y_i=\beta_0+\beta_1x_i+\varepsilon_i$: (a) show that $h_{ii}=1/n+(x_i-\bar x)^2/\{(n-1)s_x^2\}$; (b) show that $\bar h=2/n$; (c) if $h_{ii}=6/n$, how many standard deviations is $x_i$ from the mean?

**Solution.** (a) Proposition 4.12 with $k=1$: $\mathbf{S}=s_x^2$ and $\Delta_i^2=(x_i-\bar x)^2/s_x^2$, so $h_{ii}=1/n+(x_i-\bar x)^2/\{(n-1)s_x^2\}$. (b) $\sum_i(x_i-\bar x)^2=(n-1)s_x^2$, so $\sum_ih_{ii}=1+1=2$ and $\bar h=2/n$, as Unit 2's theorem **properties of leverage** gives through $\operatorname{tr}(\mathbf{H})=k+1$. (c) $6/n=1/n+(x_i-\bar x)^2/\{(n-1)s_x^2\}$ gives $|x_i-\bar x|/s_x=\sqrt{5(n-1)/n}$:

In [ ]:
round(c(n20 = sqrt(5 * 19 / 20), n100 = sqrt(5 * 99 / 100), limit = sqrt(5)), 4)

$2.18$ standard deviations at $n=20$, increasing to $\sqrt5=2.24$. The leverage $6/n$ is three times the mean, FREES's high leverage cutoff, so for $k=1$ that cutoff marks points about $2.2$ standard deviations from the mean. ■

**Exercise 4.2** (FREES exercise 5.2). A regression on one explanatory variable with $n=3$ has residuals $3.181$, $-6.362$, $3.181$ and leverages $0.8333$, $0.3333$, $0.8333$. Compute the **PRESS** statistic $\sum_ie_{(i)}^2$, the sum of squared deleted residuals.

**Solution.** By Proposition 4.3(iii), $e_{(i)}=e_i/(1-h_{ii})$, so no refitting is needed:

In [ ]:
e3 <- c(3.181, -6.362, 3.181)
c(PRESS_printed_h = sum((e3 / (1 - c(0.8333, 0.3333, 0.8333)))^2),
  PRESS_exact_h = sum((e3 / (1 - c(5, 2, 5) / 6))^2), SSE = sum(e3^2),
  ratio = sum((e3 / (1 - c(0.8333, 0.3333, 0.8333)))^2) / sum(e3^2))

$\textit{PRESS}=819.3$ with the printed leverages; they are $5/6,1/3,5/6$, the leverages of $x=(-1,0,1)$ by Exercise 4.1(a), which give $819.6$. *PRESS* is $n\,\mathrm{CV}_{(n)}$ of Unit 1, and it is $13.5$ times $\mathrm{SSE}=60.7$ here because the two outer points have $1-h_{ii}=1/6$. ■

**Exercise 4.3** (FREES exercise 5.3, parts (a), (c)). `UNLifeExpectancy`. (a) Plot life expectancy against the gross domestic product and against private expenditure on health; why are the logarithms `lnGDP` and `lnHEALTH` desirable? (c) For the regression of `LIFEEXP` on `FERTILITY`, `PUBLICEDUCATION` and `lnHEALTH`: (i) give histograms of the standardised residuals and leverages; (ii) find the standardised residual and leverage of Lesotho: an outlier, a high leverage point, or both? (iii) refit without Lesotho and compare.

**Solution.** (a)

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.3)
un <- read.csv("data/frees/csv/UNLifeExpectancy.csv")
un$COUNTRY <- gsub(" ", " ", iconv(un$COUNTRY, "latin1", "UTF-8"))
un$lnGDP <- log(un$GDP); un$lnHEALTH <- log(un$PRIVATEHEALTH)
vs <- c("GDP", "lnGDP", "PRIVATEHEALTH", "lnHEALTH")
round(rbind(skewness = sapply(un[vs], function(x) skew(na.omit(x))),
            cor_LIFEEXP = sapply(un[vs], function(x) cor(x, un$LIFEEXP, use = "complete.obs"))), 3)
op <- par(mfrow = c(1, 4), mar = c(3.2, 2.8, 0.6, 0.4), mgp = c(1.9, 0.6, 0))
for (v in vs) plot(un[[v]], un$LIFEEXP, xlab = v, ylab = if (v == "GDP") "LIFEEXP" else "", pch = 16, cex = 0.4)
par(op)

`GDP`, in billions of dollars, has skewness $9.26$: most countries are compressed against the left edge by a few large economies, and the correlation with life expectancy is $0.21$. Its logarithm has skewness $0.23$, spreads the countries evenly, and shows an increasing, nearly linear relation with correlation $0.48$. Private health expenditure, a percentage of GDP, is right-skewed ($1.16$); its logarithm is mildly left-skewed ($-0.685$) and neither version is linearly related to life expectancy (correlations $-0.04$ and $-0.03$). The logarithm also reduces the leverage of the largest values (Proposition 4.12).

(c)

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.2)
d_un <- na.omit(un[, c("COUNTRY", "LIFEEXP", "FERTILITY", "PUBLICEDUCATION", "lnHEALTH")])
fit_un <- lm(LIFEEXP ~ FERTILITY + PUBLICEDUCATION + lnHEALTH, data = d_un)
op <- par(mfrow = c(1, 2), mar = c(3.2, 3.4, 0.6, 0.6), mgp = c(1.9, 0.6, 0))
hist(rstandard(fit_un), breaks = 20, main = "", xlab = "standardised residual")
hist(hatvalues(fit_un), breaks = 20, main = "", xlab = "leverage")
par(op)
n_un <- nrow(d_un); i_les <- which(d_un$COUNTRY == "Lesotho")
c(n = n_un, r = rstandard(fit_un)[[i_les]], t = rstudent(fit_un)[[i_les]],
  h = hatvalues(fit_un)[[i_les]], cut = 3 * 4 / n_un, rank_h = rank(-hatvalues(fit_un))[[i_les]],
  D = cooks.distance(fit_un)[[i_les]], expected_flags = n_un * flag(n_un, 3),
  flags = sum(abs(rstandard(fit_un)) > 2))
d_un$COUNTRY[order(-abs(rstandard(fit_un)))[1:6]]
fit_un2 <- update(fit_un, data = d_un[-i_les, ])
round(rbind(with = coef(summary(fit_un))[, 1], without = coef(summary(fit_un2))[, 1],
            t_with = coef(summary(fit_un))[, 3], t_without = coef(summary(fit_un2))[, 3]), 3)
round(c(R2_with = summary(fit_un)$r.squared, R2_without = summary(fit_un2)$r.squared,
        s_with = summary(fit_un)$sigma, s_without = summary(fit_un2)$sigma), 3)

(i) On the $152$ countries with complete data, the standardised residuals have a long lower tail, and the leverages are concentrated near their mean with a few large values. (ii) Lesotho has $r=-3.25$ ($t=-3.36$) and leverage $0.139$, the largest of all and above $3(k+1)/n=0.079$: it is both an outlier and a high leverage point, with Cook's distance $0.43$. Six countries have $|r|>2$, against $6.9$ expected under a correct model (Proposition 4.7), and the five largest $|r|$ are all countries of southern Africa. (iii) Without Lesotho, the coefficient of `PUBLICEDUCATION` changes sign, from $-0.185$ to $0.140$, significant in neither fit; `FERTILITY` moves from $-5.40$ to $-5.31$ and `lnHEALTH` from $-1.03$ to $-1.26$; $R^2$ rises from $0.645$ to $0.659$ and $s$ falls from $6.65$ to $6.43$. ■

**Exercise 4.4** (FREES exercise 5.4, parts (a)–(c)). The $275$ term life purchasers: regress `LNFACE` on `LNINCOME`, `EDUCATION`, `NUMHH`, `MARSTAT`, `AGE` and `GENDER`. (a) Collinearity: (i) explain collinearity and a variance inflation factor; (ii) what is a large one? (iii) what can be done about it? (iv) with a table of correlations, is collinearity an issue here? (b) Unusual points: (i) define leverage; (ii) give the rules of thumb and identify unusual points; (iii) how would the leverages change with `FACE` as the response? (c) Residual analysis: plot residuals against fitted values, a normal quantile plot, and residuals against leverages; what does each show?

**Solution.** `MARSTAT` enters as the indicators `MAR1` (married) and `MAR2` (living with a partner), as in Unit 3, so $k=7$.

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
term <- subset(read.csv("data/frees/csv/TermLife.csv"), FACE > 0)
term$LNFACE <- log(term$FACE); term$LNINCOME <- log(term$INCOME)
term$MAR1 <- as.numeric(term$MARSTAT == 1); term$MAR2 <- as.numeric(term$MARSTAT == 2)
f6 <- lm(LNFACE ~ LNINCOME + EDUCATION + NUMHH + MAR1 + MAR2 + AGE + GENDER, data = term)
round(coef(summary(f6)), 4)
round(car::vif(f6), 3)
v_term <- c("LNINCOME", "EDUCATION", "NUMHH", "MAR1", "MAR2", "AGE", "GENDER")
round(cor(term[v_term]), 3)
n6 <- nobs(f6); h6 <- hatvalues(f6); n6
c(cut_3 = 3 * 8 / n6, n_above = sum(h6 > 3 * 8 / n6), max_h = max(h6),
  max_D = max(cooks.distance(f6)), expected_flags = n6 * flag(n6, 7),
  flags = sum(abs(rstandard(f6)) > 2), min_r = min(rstandard(f6)), max_abs_t = max(abs(rstudent(f6))),
  bonferroni_cut = qt(1 - 0.05 / (2 * n6), n6 - 7 - 2))
term[names(which.max(h6)), c("INCOME", "EDUCATION", "NUMHH", "MARSTAT", "AGE", "GENDER", "FACE")]
round((unlist(term[names(which.max(h6)), v_term]) - colMeans(term[v_term])) / sapply(term[v_term], sd), 2)
c(min_AGE = min(term$AGE), n_MARSTAT_2 = sum(term$MARSTAT == 2)); sort(term$INCOME)[1:4]
round(h6[term$INCOME == 260], 3)
c(max_leverage_change_with_FACE = max(abs(hatvalues(update(f6, FACE ~ .)) - h6)))
c(nR2 = unname(lmtest::bptest(f6)$statistic), p = lmtest::bptest(f6)$p.value)

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.3)
op <- par(mfrow = c(1, 3), mar = c(3.2, 3.2, 0.6, 0.4), mgp = c(1.9, 0.6, 0))
plot(fitted(f6), rstandard(f6), xlab = "fitted", ylab = "standardised residual", pch = 16, cex = 0.4)
abline(h = c(-2, 0, 2), lty = c(3, 2, 3))
qqnorm(rstandard(f6), main = "", pch = 16, cex = 0.4); qqline(rstandard(f6))
plot(h6, rstandard(f6), xlab = "leverage", ylab = "", pch = 16, cex = 0.4)
abline(v = 3 * 8 / n6, lty = 2)
par(op)

*Figure 4.10.* Standardised residuals against fitted values, their normal quantile plot, and against leverage, for the regression of `LNFACE` on seven variables.

(a)(i) Collinearity is a near-linear relation among the explanatory variables; $\mathrm{VIF}_j=1/(1-R_j^2)$ is the factor by which it multiplies $\operatorname{var}(\hat\beta_j)$ relative to uncorrelated variables (Unit 3's corollary **the standard error through the other explanatory variables** and Proposition 4.20). (ii) FREES calls $\mathrm{VIF}_j>10$ severe, equivalently $R_j^2>0.9$; ISLR uses $5$ or $10$. (iii) Drop one of the related variables, combine them into one, replace one by a variable carrying similar information, or, for a polynomial, centre before squaring (Proposition 4.21). (iv) The largest factor is $2.57$, for `MAR1`, and the largest correlation is $0.62$, between `MAR1` and `GENDER`: collinearity is not what makes `MAR1`, `MAR2` and `AGE` insignificant.

(b)(i) The leverage $h_{ii}$ is the $i$-th diagonal entry of $\mathbf{H}$, the weight of $y_i$ in its own fitted value, and by Proposition 4.12 a function of the Mahalanobis distance of $\mathbf{x}_i$ from the centre of the design. (ii) FREES flags $h_{ii}>3(k+1)/n=0.087$; $17$ households exceed it. The largest, $0.168$, belongs to the youngest respondent, aged $20$, one of the ten households with `MARSTAT` $2$ and the third-lowest income, \$500: it lies $3.81$ standard deviations below the mean in `LNINCOME`, $5.14$ above in `MAR2` and $2.26$ below in `AGE`, unusual jointly as Proposition 4.12 measures; the two households with income \$260 have leverages $0.119$ and $0.107$. No Cook's distance exceeds $0.15$. (iii) Leverages depend on $\mathbf{X}$ alone, so with `FACE` as the response they are unchanged, as the zero difference confirms.

(c) Against the fitted values the standardised residuals show no trend; $11$ exceed $2$ in absolute value, against $12.5$ expected (Proposition 4.7). The quantile plot follows the line except in the lower tail, where the smallest residual is $-3.89$; its studentised residual, $3.99$ in absolute value, exceeds the Bonferroni cutoff $3.80$ of Corollary 4.6. Koenker's statistic $17.8$ on $7$ degrees of freedom ($p=0.013$) indicates some heteroscedasticity that the plot does not show. Against leverage, no point combines a large residual with a large leverage, as the small Cook's distances confirm. ■

**Exercise 4.5** (ISLR §3.7, exercise 14). (a) After `set.seed(1)`, generate $x_1$ uniform on $(0,1)$, $x_2=0.5x_1+\delta/10$ and $y=2+2x_1+0.3x_2+\varepsilon$ for $100$ observations, with $\delta$ and $\varepsilon$ standard normal (the commands open the solution); write the model and its coefficients. (b) What is the correlation of `x1` and `x2`? Plot them. (c) Regress `y` on both: compare the estimates with the true coefficients; can $H_0\colon\beta_1=0$ and $H_0\colon\beta_2=0$ be rejected? (d) Regress on `x1` only; (e) on `x2` only. (f) Do (c)–(e) contradict each other? (g) Add the mismeasured observation `x1` $=0.1$, `x2` $=0.8$, `y` $=6$ and refit (c)–(e): is it an outlier, a high leverage point, or both, in each model?

**Solution.** (a) $y=\beta_0+\beta_1x_1+\beta_2x_2+\varepsilon$ with $\beta_0=2$, $\beta_1=2$, $\beta_2=0.3$ and $\varepsilon\sim N(0,1)$.

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.2)
set.seed(1)
x1 <- runif(100); x2 <- 0.5 * x1 + rnorm(100) / 10; y <- 2 + 2 * x1 + 0.3 * x2 + rnorm(100)
op <- par(mar = c(3.2, 3.4, 0.6, 0.6), mgp = c(1.9, 0.6, 0))
plot(x1, x2, pch = 16, cex = 0.5)
par(op)
c(cor_x1_x2 = cor(x1, x2), VIF = 1 / (1 - cor(x1, x2)^2))
fits14 <- list(both = lm(y ~ x1 + x2), x1_only = lm(y ~ x1), x2_only = lm(y ~ x2))
lapply(fits14, function(m) round(coef(summary(m)), 4))

(b) The correlation is $0.835$, and the scatter plot is an increasing band. (c) $\hat\beta_0=2.13$, $\hat\beta_1=1.44$ and $\hat\beta_2=1.01$: $\beta_0$ is well estimated, $\beta_1$ and $\beta_2$ poorly, with standard errors $0.72$ and $1.13$ inflated by $\mathrm{VIF}=3.30$ (Proposition 4.20). $H_0\colon\beta_1=0$ is rejected at $5\%$, barely ($p=0.049$); $H_0\colon\beta_2=0$ is not ($p=0.38$). (d) Alone, `x1` has slope $1.98$ with $t=4.99$, and (e) `x2` has slope $2.90$ with $t=4.58$: both reject $H_0$. (f) No: the simple slope of `x2` estimates $\beta_2$ plus $\beta_1$ times the slope of `x1` on `x2` (Unit 3's omitted variable theorem), which is far from $0$; and in (c) each variable is tested given the other, with standard errors inflated by the collinearity (Proposition 4.18), so each $t$ ratio is small although the two jointly matter.

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
x1 <- c(x1, 0.1); x2 <- c(x2, 0.8); y <- c(y, 6)
fits14g <- list(both = lm(y ~ x1 + x2), x1_only = lm(y ~ x1), x2_only = lm(y ~ x2))
lapply(fits14g, function(m) round(coef(summary(m)), 4))
round(sapply(fits14g, function(m) c(t_101 = rstudent(m)[[101]], h_101 = hatvalues(m)[[101]],
  h_over_mean = hatvalues(m)[[101]] / (length(coef(m)) / 101),
  cut = 3 * length(coef(m)) / 101, D_101 = cooks.distance(m)[[101]])), 4)
c(cor_after = cor(x1, x2), VIF_after = 1 / (1 - cor(x1, x2)^2))

(g) In the regression on both, the new point has leverage $0.415$, fourteen times the mean and far above $3(k+1)/n=0.089$, and $t=2.11$: a high leverage point and a mild outlier, with Cook's distance $1.02$. It lies off the band of the other points (Proposition 4.12), lowers the correlation to $0.739$, and reverses the conclusions of (c): $\hat\beta_1=0.54$ is no longer significant while $\hat\beta_2=2.51$ is ($p=0.006$). On `x1` alone its leverage $0.033$ is below $0.059$ but $t=3.44$: an outlier. On `x2` alone its leverage $0.101$ exceeds $0.059$ and $t=1.14$: a high leverage point, not an outlier. ■

**Exercise 4.6** (SOA sample question 14). The spread of the residuals of an ordinary least squares fit increases with the predicted values, and the observed responses range from $0$ to $100$. Which transformations of the response may make the variance of the residuals more constant? I. The logarithm of one plus the response. II. The square root. III. The logit. (A) None; (B) I and II only; (C) I and III only; (D) II and III only; (E) the correct answer is not given by (A), (B), (C) or (D).

**Solution.** $g(y)=\ln(1+y)$ and $g(y)=\sqrt y$ are increasing and concave on $[0,\infty)$, with increasing convex inverses $h(t)=e^t-1$ and $h(t)=t^2$ on $g([0,\infty))$. By Proposition 4.36, if $g(y_i)$ follows a linear model with identically distributed errors, the variance of $y_i$ is nondecreasing in its mean, the pattern described, and the transformation returns the model to constant variance; so I and II are true. $\ln(1+y)$ is defined at $y=0$, where $\ln y$ is not. III is false: $\operatorname{logit}(y)=\ln\{y/(1-y)\}$ is defined only for $0<y<1$, and even for $y/100$ it is infinite at the observed values $0$ and $100$. I and II only: (B). ■